# Radar Tracking Benchmark Implementation Notebook

This notebook mirrors the current single-target radar tracking workflow in `Radar_Tracking_Benchmark`.

Current implementation focus:
1. Single-target radar tracking
2. Synthetic benchmark using `RANGE_BEARING` measurements
3. Comparison of `EKF`, `KalmanNet`, and `BiMambaTrackerModel`
4. Default benchmark context `seq_len = 100`
5. Archived multi-target and association code kept outside the active workflow

Generated from the current project sources on 2026-04-24.


## Project Overview

The active benchmark now uses the shared `BiMambaTracker` model together with the long-sequence radar generator in this folder.

The current workflow includes:
- the synthetic single-target data generator from `data_generation.py`
- the classical `EKF` baseline
- the `KalmanNet` baseline with updated checkpoint selection
- the shared `BiMambaTrackerModel` imported from `../../BiMambaTracker/models/tracker_model.py`
- a compact benchmark runner in `benchmark_mamba.py`

The main benchmark path is the `seq_len = 100` setting, while longer contexts remain optional experiments.


### Current Workflow Summary


# Long-Sequence Radar Benchmark

This project is currently organized around a clean single-target benchmark:

- single-target radar tracking only
- synthetic data generated from the provided long-sequence code
- `RANGE_BEARING` observations
- default benchmark context `seq_len = 100`
- optional longer-context experiments through `SIM_SEQ_LEN`
- CUDA execution for the BiMamba path

Compared methods:

- `EKF` from the provided codebase
- `KalmanNet` from the provided codebase
- `BiMambaTrackerModel` from `../../BiMambaTracker/models/tracker_model.py`

Helpful commands:

```bash
../../.venv/bin/python benchmark_mamba.py
```

```bash
bash run_full_benchmark.sh
```

```bash
SIM_SEQ_LEN=300 KNET_TRAIN=1 ../../.venv/bin/python benchmark_mamba.py
```


## Simulation Setup

Core simulation parameters, the constant-acceleration state-space model, and the long-context trajectory generator used by the benchmark.


### sim_params.py

**File:** `sim_params.py`


In [ ]:
import os
import torch


# Parameters used for the simulation
class simparams_class:
    def __init__(self):
        self.use_cuda           = torch.cuda.is_available()
        self.dt                 = 1
        self.n_batch            = 64
        self.in_mult_KNet       = 5 #determines the number of paramters Knet
        self.out_mult_KNet      = 40 #determines the number of paramters Knet
        self.learning_rate      = 1e-3
        self.weight_decay       = 1e-4
        # Default to the original long-sequence reference setting.
        self.seq_len            = int(os.environ.get("SIM_SEQ_LEN", 100))
        self.q2_scale           = 1
        self.r2_scale           = 1
        self.meas_type          = "RANGE_BEARING" #FULL/POS_ONLY/RANGE_BEARING
        self.init_type          = "MEAS" #MEAS/GTS
        self.stopping_criteria  = 20
        self.update_thr_db      = 0.01
        self.dataset_size       = int(os.environ.get("SIM_DATASET_SIZE", 512))
        self.epochs             = 1000
        self.train_knet         = True
        self.knet_name          = "best_knet.pt"
        self.knet_rb_wrap_theta = os.environ.get("KNET_RB_WRAP_THETA", "1").strip().lower() in ("1", "true", "yes", "y")
        self.knet_rb_arc_length = os.environ.get("KNET_RB_ARC_LENGTH", "1").strip().lower() in ("1", "true", "yes", "y")
        self.noise_ref          = "FIXED" #SNR/FIXED
        self.range_std          = 30 #in m
        self.bearing_std        = 1 # in degrees
      

def validate_simparams(sim_params):
    assert sim_params.meas_type in ("FULL", "POS_ONLY", "RANGE_BEARING"), "Invalid measurement type argument"
    assert sim_params.init_type in ("MEAS", "GTS"), "Invalid init_type type argument"
    assert sim_params.noise_ref in ("FIXED", "SNR"), "Invalid noise_ref type argument"

        


### data_model.py

**File:** `data_model.py`


In [ ]:
####################################
'''
Data model - constant acceleration
'''
####################################

import torch
import sys
import math

TWO_PI = 2.0 * math.pi

def wrap_to_2pi(angle):
    # [0, 2pi)
    return torch.remainder(angle, TWO_PI)

class ConstantAcceleration2DModel:

    def __init__(self, sim_params):

        # State: [x, vx, ax, y, vy, ay]
        self.m = 6
        self.dt = sim_params.dt
        self.meas_type = sim_params.meas_type

        # Discrete time CA dynamics
        self.F = torch.tensor([
            [1,     self.dt,   0.5*self.dt**2,  0,      0,              0],
            [0,     1,         self.dt,         0,      0,              0],
            [0,     0,         1,               0,      0,              0],
            [0,     0,         0,               1,      self.dt,        0.5*self.dt**2],
            [0,     0,         0,               0,      1,              self.dt],
            [0,     0,         0,               0,      0,              1]
        ])

        # Measurement model
        if self.meas_type == 'FULL':
            self.n = 6
            self.H = torch.eye(self.n)

        elif self.meas_type == 'POS_ONLY':
            self.n = 2
            self.H = torch.zeros(self.n, self.m)
            self.H[0, 0] = 1.0  # x
            self.H[1, 3] = 1.0  # y

        elif self.meas_type == 'RANGE_BEARING':
            self.n = 2
            self.H = None  # nonlinear

        else:
            print("Unknown meas_type")
            sys.exit(1)

        # initializations for KNet
        self.prior_Q     = torch.diag(torch.ones(self.m))
        self.prior_Sigma = torch.diag(torch.ones(self.m))
        self.prior_S     = torch.diag(torch.ones(self.n))

    def f(self, x):
        F = self.F.to(x.device)
        return F.unsqueeze(0) @ x

    def h(self, x):
        if self.meas_type in ("FULL", "POS_ONLY"):
            H = self.H.to(x.device)
            return H.unsqueeze(0) @ x

        px = x[:, 0, 0]
        py = x[:, 3, 0]
        r = torch.sqrt(px*px + py*py + 1e-12)
        theta = torch.atan2(py, px)          # [-pi, pi]
        theta = wrap_to_2pi(theta)           # [0, 2pi)

        y = torch.stack([r, theta], dim=1).unsqueeze(-1)
        return y

    def H_jacobian(self, x):
        B = x.shape[0]

        if self.meas_type in ("FULL", "POS_ONLY"):
            H = self.H.to(x.device)
            return H.unsqueeze(0).repeat(B, 1, 1)

        px = x[:, 0, 0]
        py = x[:, 3, 0]
        r2 = px*px + py*py + 1e-12
        r = torch.sqrt(r2)

        H = torch.zeros(B, 2, self.m, device=x.device)
        H[:, 0, 0] = px / r
        H[:, 0, 3] = py / r
        H[:, 1, 0] = -py / r2
        H[:, 1, 3] =  px / r2
        return H


### data_generation.py

**File:** `data_generation.py`


In [ ]:
"""
Generate synthetic trajectories and observations for the Constant-Acceleration model.

Noise distributions - Gaussian and Laplacian (double-exponential) noise for measurement 
and process noise.
"""

import torch
from torch.distributions.multivariate_normal import MultivariateNormal
import sys
import random
import math

TWO_PI = 2.0 * math.pi

# Noise selection flags, Choose: "gaussian" | "laplace"
MEAS_NOISE_DIST = "gaussian"
PROC_NOISE_DIST = "gaussian"

# compute coarse velocity from first two measurements for initialisation
RB_INIT_VEL_FROM_TWO_MEAS = True

RB_INIT_VEL_CLIP = 300.0

REFERENCE_SEQ_LEN = 100.0

def wrap_to_2pi(angle):
    return torch.remainder(angle, TWO_PI)


def get_motion_profile(sim_params):
    """
    Keep the long-context benchmark in the same physical regime as the original
    100-step setup.

    With constant-acceleration dynamics, extending the context by 10x increases
    the velocity-driven displacement by 10x and the acceleration-driven
    displacement by 100x if we keep the same initial state ranges. That makes
    the 1000-step trajectories numerically extreme and unfair to models trained
    on the original regime. We therefore shrink the velocity, acceleration, and
    process-noise scales as the sequence gets longer.
    """
    seq_len = max(float(sim_params.seq_len), 1.0)
    seq_ratio = max(seq_len / REFERENCE_SEQ_LEN, 1.0)

    vel_scale = 1.0 / seq_ratio
    acc_scale = 1.0 / (seq_ratio ** 2)

    return {
        "pos_min": 20000.0,
        "pos_max": 30000.0,
        "vel_min": -200.0 * vel_scale,
        "vel_max": 200.0 * vel_scale,
        "accn_min": -3.0 * acc_scale,
        "accn_max": 3.0 * acc_scale,
        "accn_rms_q": 1.0 * acc_scale,
    }


def Q_CA_1D(dt, q):
    dt = torch.as_tensor(dt)
    dt2, dt3, dt4, dt5 = dt**2, dt**3, dt**4, dt**5
    Q = torch.tensor([[dt5/20, dt4/8, dt3/6],
                      [dt4/8,  dt3/3, dt2/2],
                      [dt3/6,  dt2/2, dt     ]])
    return q * Q


def Q_CA_2D(dt, qx, qy):
    return torch.block_diag(Q_CA_1D(dt, qx),
                            Q_CA_1D(dt, qy))


def sample_laplace_like(std_dev_vec: torch.Tensor, num_batch: int, device=None):
    """
    Sample iid Laplace(0, b) noise with per-dimension std given by std_dev_vec.
    For Laplace(0,b): std = sqrt(2)*b  =>  b = std/sqrt(2)
    """
    if device is None:
        device = std_dev_vec.device
    std = std_dev_vec.to(device)                       # [D]
    b = std / torch.sqrt(torch.tensor(2.0, device=device))
    D = std.numel()

    U = torch.rand(num_batch, D, device=device) - 0.5
    X = -b * torch.sign(U) * torch.log1p(-2.0 * torch.abs(U))
    return X.unsqueeze(-1)                             # [B, D, 1]


def generate_synthetic_data(model, sim_params, snr_dB, seq_mul_fact=1, batch_size=None):

    num_batch = sim_params.dataset_size if batch_size is None else batch_size
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    motion_profile = get_motion_profile(sim_params)

    # Initial state values
    pos_min, pos_max = motion_profile["pos_min"], motion_profile["pos_max"]
    vel_min, vel_max = motion_profile["vel_min"], motion_profile["vel_max"]
    accn_min, accn_max = motion_profile["accn_min"], motion_profile["accn_max"]

    accn_rms_q = motion_profile["accn_rms_q"]  # to compute process noise

    pos_var = (pos_max - pos_min)**2 / 12
    vel_var = (vel_max - vel_min)**2 / 12
    accn_var = (accn_max - accn_min)**2 / 12

    snr_linear = 10**(snr_dB/10)

    pos_noise_var = pos_var / snr_linear
    vel_noise_var = vel_var / snr_linear
    accn_noise_var = accn_var / snr_linear

    # error RMS
    pos_rms = pos_noise_var ** 0.5
    vel_rms = vel_noise_var ** 0.5
    accn_rms = accn_noise_var ** 0.5

    # process noise (jerk variance)
    jerk_var = (accn_rms_q ** 2) / model.dt
    process_noise = Q_CA_2D(model.dt, jerk_var, jerk_var)
    Q = process_noise * sim_params.q2_scale

    if sim_params.meas_type == 'POS_ONLY':
        std_dev_vec = torch.tensor([pos_rms, pos_rms], device=device)

    elif sim_params.meas_type == 'FULL':
        std_dev_vec = torch.tensor([pos_rms, vel_rms, accn_rms,
                                    pos_rms, vel_rms, accn_rms], device=device)

    elif sim_params.meas_type == 'RANGE_BEARING':
        if sim_params.noise_ref == "SNR":
            r_typical = ((pos_min + pos_max) / 2.0)
            range_rms = pos_rms
            bearing_rms = pos_rms / r_typical
        else:
            range_rms = sim_params.range_std
            bearing_rms = sim_params.bearing_std * torch.pi / 180.0

        std_dev_vec = torch.tensor([range_rms, bearing_rms], device=device)
        print(f"Range rms = {range_rms:.3f} m, Bearing rms = {bearing_rms*180/torch.pi:.3f} deg")

    else:
        print("model.meas_type must be FULL, POS_ONLY, or RANGE_BEARING")
        sys.exit(1)

    R = torch.diag(std_dev_vec**2) * sim_params.r2_scale

    proc_std_vec = torch.sqrt(torch.diag(Q)).to(device)  # [m]

    # Gaussian distributions (kept for gaussian mode with full covariance)
    meas_dist = MultivariateNormal(loc=torch.zeros(model.n, device=device), covariance_matrix=R.to(device))
    proc_dist = MultivariateNormal(loc=torch.zeros(model.m, device=device), covariance_matrix=Q.to(device))

    # one more iteration is used to generate an initialisation vector
    L = int(sim_params.seq_len * seq_mul_fact) + 1
    x_seq = torch.zeros(num_batch, model.m, L, device=device)
    y_seq = torch.zeros(num_batch, model.n, L, device=device)

    # init state
    x_init = torch.zeros(num_batch, model.m, 1, device=device)
    x_init[:, 0, 0] = pos_min + (pos_max - pos_min) * torch.rand(num_batch, device=device)
    x_init[:, 1, 0] = vel_min + (vel_max - vel_min) * torch.rand(num_batch, device=device)
    x_init[:, 2, 0] = accn_min + (accn_max - accn_min) * torch.rand(num_batch, device=device)
    x_init[:, 3, 0] = pos_min + (pos_max - pos_min) * torch.rand(num_batch, device=device)
    x_init[:, 4, 0] = vel_min + (vel_max - vel_min) * torch.rand(num_batch, device=device)
    x_init[:, 5, 0] = accn_min + (accn_max - accn_min) * torch.rand(num_batch, device=device)

    init_vec = None
    x_prev = x_init

    for t in range(L):
 
        # state evolution
        x_t = model.f(x_prev)

        # process noise
        if PROC_NOISE_DIST.lower() == "gaussian":
            eq = proc_dist.rsample((num_batch,)).unsqueeze(-1)  # [B,m,1]
        elif PROC_NOISE_DIST.lower() == "laplace":
            eq = sample_laplace_like(proc_std_vec, num_batch, device=device)  # [B,m,1]
        else:
            raise ValueError(f"Unknown PROC_NOISE_DIST: {PROC_NOISE_DIST}")

        x_t = x_t + eq

        # measurement
        y_t = model.h(x_t)

        # measurement noise
        meas_std_vec = torch.sqrt(torch.diag(R)).to(device)  # [n]

        if MEAS_NOISE_DIST.lower() == "gaussian":
            er = meas_dist.rsample((num_batch,)).unsqueeze(-1)  # [B,n,1]
        elif MEAS_NOISE_DIST.lower() == "laplace":
            er = sample_laplace_like(meas_std_vec, num_batch, device=device)  # [B,n,1]
        else:
            raise ValueError(f"Unknown MEAS_NOISE_DIST: {MEAS_NOISE_DIST}")

        y_t = y_t + er

        # wrap bearing to [0, 2pi) for RANGE_BEARING
        if sim_params.meas_type == 'RANGE_BEARING':
            y_t[:, 1, 0] = wrap_to_2pi(y_t[:, 1, 0])

        x_seq[:, :, t] = x_t.squeeze(-1)
        y_seq[:, :, t] = y_t.squeeze(-1)
        x_prev = x_t

        # init vector (MEAS) computed once when we have at least t>=1
        if (init_vec is None) and (sim_params.init_type == 'MEAS') and (t >= 1):
            init_vec = torch.zeros_like(x_init)

            if sim_params.meas_type in ("FULL", "POS_ONLY"):
                for idx in range(model.n):
                    col_idx = (model.H[idx, :] == 1).nonzero(as_tuple=True)[0]
                    init_vec[:, col_idx, :] = y_seq[:, idx, 0].unsqueeze(-1).unsqueeze(-1)

            elif sim_params.meas_type == "RANGE_BEARING":
                r0  = y_seq[:, 0, 0]
                th0 = y_seq[:, 1, 0]
                init_vec[:, 0, 0] = r0 * torch.cos(th0)
                init_vec[:, 3, 0] = r0 * torch.sin(th0)

                if RB_INIT_VEL_FROM_TWO_MEAS:
                    r1  = y_seq[:, 0, 1]
                    th1 = y_seq[:, 1, 1]
                    x0 = init_vec[:, 0, 0]
                    y0 = init_vec[:, 3, 0]
                    x1 = r1 * torch.cos(th1)
                    y1 = r1 * torch.sin(th1)
                    vx0 = (x1 - x0) / model.dt
                    vy0 = (y1 - y0) / model.dt
                    if RB_INIT_VEL_CLIP is not None:
                        vx0 = torch.clamp(vx0, -RB_INIT_VEL_CLIP, RB_INIT_VEL_CLIP)
                        vy0 = torch.clamp(vy0, -RB_INIT_VEL_CLIP, RB_INIT_VEL_CLIP)
                    init_vec[:, 1, 0] = vx0
                    init_vec[:, 4, 0] = vy0
                # ax, ay is initialised to zero

                init_vec = init_vec.unsqueeze(-1) if init_vec.dim() == 2 else init_vec
            else:
                print("Unknown meas_type in init")
                sys.exit(1)

        elif sim_params.init_type == 'GTS':
            init_vec = x_seq[:, :, 0].unsqueeze(-1)

    x_seq = x_seq[:, :, 1:]
    y_seq = y_seq[:, :, 1:]

    return x_seq, y_seq, init_vec, Q, R


def split_data(x_true, y_obs, init_state, train_frac=0.8, val_frac=0.2):
    N = x_true.shape[0]
    idxs = list(range(N))
    random.shuffle(idxs)
    n_train = int(N * train_frac)
    n_val = int(N * val_frac)
    train_idx = idxs[:n_train]
    val_idx = idxs[n_train:n_train+n_val]

    def select(idxs):
        return x_true[idxs], y_obs[idxs], init_state[idxs]

    return select(train_idx), select(val_idx)


## Single-Target Baselines

Classical and neural baseline trackers used in the current benchmark. `knet_nn_arch1.py` is kept as an alternative KalmanNet architecture snapshot, while `knet_nn.py` is the main baseline used by the active scripts.


### ekf.py

**File:** `ekf.py`


In [ ]:
####################################
'''
EKF implementation
'''
####################################
import torch
import sys
import math


def wrap_to_pi(angle):
    return (angle + math.pi) % (2.0 * math.pi) - math.pi


class extended_kf:
    def __init__(self, model, Q, R, sim_params):

        self.model = model
        self.meas_type = sim_params.meas_type
        
        dataset_size = sim_params.dataset_size
        
        self.Q = torch.zeros(dataset_size, self.model.m, self.model.m)
        self.R = torch.zeros(dataset_size, self.model.n, self.model.n)

        for bidx in range(dataset_size):
            self.Q[bidx,:,:] = Q
            self.R[bidx,:,:] = R

        self.x = torch.zeros(dataset_size,self.model.m,1)
        self.P = torch.zeros(dataset_size, self.model.m, self.model.m)

    def reset(self, x_init):
        self.x[:,:,0] = x_init.squeeze(-1)
        
        if self.meas_type == "FULL":
            init_cov = self.R[0,:,:]
        elif self.meas_type == "POS_ONLY":
            init_cov = torch.eye(self.model.m)*1e4
            for i in range(self.model.n):
                cols = (self.model.H[i] == 1).nonzero(as_tuple=False)
                j = cols[0].item()
                init_cov[j,j] = self.R[0,i,i]
        elif self.meas_type == "RANGE_BEARING":
            # large uncertainty
            init_cov = torch.eye(self.model.m) * 1e4

            # use current init position to estimate (r,theta)
            px = self.x[:, 0, 0]
            py = self.x[:, 3, 0]
            r2 = px*px + py*py + 1e-12
            r = torch.sqrt(r2)

            sig_r2 = self.R[0,0,0]
            sig_th2 = self.R[0,1,1]

            # Var(x) = cos^2 th * sig_r2 + (r^2 sin^2 th) * sig_th2
            # Var(y) = sin^2 th * sig_r2 + (r^2 cos^2 th) * sig_th2
            th = torch.atan2(py, px)

            var_x = (torch.cos(th)**2) * sig_r2 + (r**2) * (torch.sin(th)**2) * sig_th2
            var_y = (torch.sin(th)**2) * sig_r2 + (r**2) * (torch.cos(th)**2) * sig_th2

            for b_idx in range(self.x.shape[0]):
                init_cov_b = init_cov.clone()
                init_cov_b[0,0] = var_x[b_idx].item()
                init_cov_b[3,3] = var_y[b_idx].item()
                self.P[b_idx,:,:] = init_cov_b
            self.K = 0; self.S = 0; self.nu = 0
            return
        else:
            print("Unknown meas_type")
            sys.exit(-1)

        
        for b_idx in range(x_init.shape[0]):
            self.P[b_idx,:,:] = init_cov             
        
        self.K = 0
        self.S = 0
        self.nu = 0

    def f(self, x):
        return self.model.f(x)

    def h(self, x):
        return self.model.h(x)

    def predict(self):
        self.x = self.f(self.x)
        self.P = self.model.F @ self.P @ self.model.F.T + self.Q

    def update(self, z):
        z_pred = self.h(self.x).squeeze(-1)   # [B,n]
        H = self.model.H_jacobian(self.x)     # [B,n,m]
        y = (z - z_pred)                      # [B,n]

        # wrap bearing residual to [-pi, pi)
        if self.meas_type == "RANGE_BEARING":
            y[:, 1] = wrap_to_pi(y[:, 1])

        y = y.unsqueeze(-1)                   # [B,n,1]
        self.nu = y.squeeze(-1)  
        
        temp = torch.matmul(H, self.P)
        self.S = torch.matmul(temp, torch.transpose(H,1,2)) + self.R
        
        temp = torch.matmul(self.P, torch.transpose(H,1,2))
        self.K = torch.matmul(temp, torch.linalg.inv(self.S))
        
        self.x = self.x + torch.matmul(self.K, y)
        
        I = torch.eye(self.model.m).expand_as(self.P)                 # [B, m, m]
        IKH = I - torch.matmul(self.K, H)                                           # [B, m, m]
        KRKT = torch.matmul(self.K, torch.matmul(self.R, self.K.transpose(1, 2)))   # [B, m, m]
        self.P = torch.matmul(torch.matmul(IKH, self.P), IKH.transpose(1, 2)) + KRKT
           
    
class batch_ekf:
    def __init__(self, ekf_class):  
        self.ekf_obj = ekf_class

    def evaluate_batch(self, true_state, measurements, init_state):
        batch_size, obs_dim, seq_len = measurements.shape
        state_dim = self.ekf_obj.model.m

        estimates = torch.zeros(batch_size, state_dim, seq_len)
        P_seq     = torch.zeros(batch_size, state_dim, state_dim, seq_len)
        K_seq     = torch.zeros(batch_size, state_dim, obs_dim, seq_len)
        S_seq     = torch.zeros(batch_size, obs_dim, obs_dim, seq_len)
        NU_seq    = torch.zeros(batch_size, obs_dim, seq_len)
        
        self.ekf_obj.reset(init_state)
 
        for t_idx in range(seq_len):
            self.ekf_obj.predict()
            self.ekf_obj.update(measurements[:, :, t_idx])
        
            estimates[:, :, t_idx] = self.ekf_obj.x.squeeze(-1)
            P_seq[:, :, :, t_idx]  = self.ekf_obj.P
            K_seq[:, :, :, t_idx]  = self.ekf_obj.K
            S_seq[:, :, :, t_idx]  = self.ekf_obj.S
            NU_seq[:, :, t_idx]    = self.ekf_obj.nu

        return estimates, P_seq, K_seq, S_seq, NU_seq

def compute_lossvstime(truestate_dataset, ekf_estimates):
    ekf_loss_fn = torch.nn.MSELoss(reduction='mean')
    
    total_loss_dB = 10*torch.log10(ekf_loss_fn(truestate_dataset, ekf_estimates))
    
    seq_len = ekf_estimates.shape[2]
    
    loss_vs_time_dB = torch.zeros(seq_len)
    for idx in range(seq_len):
        loss_vs_time_dB[idx] = 10*torch.log10(ekf_loss_fn(truestate_dataset[:,:,idx], ekf_estimates[:,:,idx]))
        
    return loss_vs_time_dB, total_loss_dB
        
def mse_per_dim(est, true):
    # est/true: [B, m, T]
    mse_dim = torch.mean((est - true)**2, dim=(0, 2))   # [m]
    mse_dim_db = 10*torch.log10(mse_dim)
    return mse_dim_db
        
    
    
    



### knet_nn.py

**File:** `knet_nn.py`


In [ ]:
####################################
'''
KalmanNet neural network code adopted from KalmanNet github repo, modifications are done 
on the original code
'''
####################################

import torch
import torch.nn as nn
import torch.nn.functional as func
import math

TWO_PI = 2.0 * math.pi

def wrap_to_pi(angle):
    return (angle + math.pi) % (2.0 * math.pi) - math.pi


class KalmanNetNN(torch.nn.Module):

    ###################
    ### Constructor ###
    ###################
    def __init__(self):
        super().__init__()

    def _sanitize(self, tensor, clamp_value=100.0):
        tensor = torch.nan_to_num(
            tensor,
            nan=0.0,
            posinf=clamp_value,
            neginf=-clamp_value,
        )
        return torch.clamp(tensor, -clamp_value, clamp_value)

    def _sanitize_hidden(self, tensor):
        return self._sanitize(tensor, clamp_value=50.0)

    def _safe_normalize(self, tensor):
        tensor = self._sanitize(tensor, clamp_value=20.0)
        return func.normalize(tensor, p=2, dim=1, eps=1e-12, out=None)
        
    def apply_film(self, x, gamma, beta, gamma_scale=1, beta_scale=1):
        """
        x, gamma, beta: [1, B, D]
        Bound modulation:
          gamma = 1 + gamma_scale * tanh(gamma)
          beta  = beta_scale * tanh(beta)
        """
        
        g = 1+gamma_scale * torch.tanh(gamma)
        b = beta_scale * torch.tanh(beta)
        
        return x * g + b

        
    def UpdateFilmWt(self, controller_out):
        """
        controller_out: [batch_size, 2*(film_fc7_dim + film_fc5_dim + film_fc6_dim)]
        Produces gamma/beta for FC7, FC5, FC6 as [1, batch_size, dim]
        """
        assert controller_out.dim() == 2, f"controller_out must be [B, D], got {controller_out.shape}"
        B, D = controller_out.shape
        assert B == self.batch_size, f"Batch mismatch: controller_out B={B}, expected {self.batch_size}"
    
        fc7_dim = getattr(self, "film_fc7_dim", self.d_output_FC7)
        fc5_dim = getattr(self, "film_fc5_dim", self.d_output_FC5)
        fc6_dim = getattr(self, "film_fc6_dim", self.d_output_FC6)
    
        expected_D = 2 * (fc7_dim + fc5_dim + fc6_dim)
        assert D == expected_D, f"controller_out dim={D}, expected {expected_D} = 2*(fc7_dim+fc5_dim+fc6_dim)"
    
        split_sizes = [fc7_dim, fc7_dim, fc5_dim, fc5_dim, fc6_dim, fc6_dim]
        gamma7, beta7, gamma5, beta5, gamma6, beta6 = torch.split(controller_out, split_sizes, dim=1)
    
        self.gamma_fc7 = gamma7.unsqueeze(0)
        self.beta_fc7  = beta7.unsqueeze(0)
    
        self.gamma_fc5 = gamma5.unsqueeze(0)
        self.beta_fc5  = beta5.unsqueeze(0)
    
        self.gamma_fc6 = gamma6.unsqueeze(0)
        self.beta_fc6  = beta6.unsqueeze(0)

        
    
    def NNBuild(self, SysModel, args):

        # Device
        if args.use_cuda:
            self.device = torch.device('cuda')
        else:
            self.device = torch.device('cpu')

        self.InitSystemDynamics(SysModel.f, SysModel.h, SysModel.m, SysModel.n)

        # Number of neurons in the 1st hidden layer
        #H1_KNet = (SysModel.m + SysModel.n) * (10) * 8

        # Number of neurons in the 2nd hidden layer
        #H2_KNet = (SysModel.m * SysModel.n) * 1 * (4)

        self.InitKGainNet(SysModel.prior_Q, SysModel.prior_Sigma, SysModel.prior_S, args)
        
        self.meas_type = args.meas_type

        # ------------------------------------------------------------
        # RANGE_BEARING residual handling toggles
        #
        # Two independent features that can be enabled/disabled:
        #   1) wrap bearing residuals to [-pi, pi)
        #   2) convert bearing residual (rad) to tangential displacement (m)
        # ------------------------------------------------------------
        self.rb_wrap_theta = getattr(args, "knet_rb_wrap_theta", True)
        self.rb_arc_length = getattr(args, "knet_rb_arc_length", True)
        # The benchmark scales meters -> kilometers, so positions live around
        # 20-30 km and range can exceed 40 km. The previous 20.0 clamp was
        # cutting off valid states/measurements and made KalmanNet unusable.
        self.state_clip_value = 100.0
        self.gain_clip_value = 5.0
        self.measurement_clip_value = 100.0
        
        # ------------------------------------------------------------
        # FiLM (Feature-wise Linear Modulation)
        #
        # FiLM layers are constructed below but are intentionally
        # DISABLED for the current benchmark (self.film = False).
        #
        # Reason: the current project scope is a clean single-target
        # benchmark of EKF / KalmanNet / Mamba. The FiLM controller was
        # explored as an extension for adaptive noise estimation but is
        # not part of the reported results.
        #
        # To enable FiLM in a future experiment:
        #   1. Set self.film = True here.
        #   2. Call self.UpdateFilmWt(controller_output) before each
        #      KalmanNet step.
        #   3. Retrain from scratch, because FiLM changes the network's
        #      effective weights.
        # ------------------------------------------------------------
        self.film               = False   # disabled for current benchmark scope
        self.num_film_layers    = 3       # FC5, FC6, FC7 (unused while film=False)
        
        self.film_fc7_dim       = self.d_output_FC7
        self.film_fc5_dim       = self.d_output_FC5
        self.film_fc6_dim       = self.d_output_FC6
        
        # Identity FiLM params (on device)
        self.gamma_fc7 = torch.zeros(1, self.batch_size, self.film_fc7_dim, device=self.device)
        self.beta_fc7  = torch.zeros(1, self.batch_size, self.film_fc7_dim, device=self.device)
        
        self.gamma_fc5 = torch.zeros(1, self.batch_size, self.film_fc5_dim, device=self.device)
        self.beta_fc5  = torch.zeros(1, self.batch_size, self.film_fc5_dim, device=self.device)
        
        self.gamma_fc6 = torch.zeros(1, self.batch_size, self.film_fc6_dim, device=self.device)
        self.beta_fc6  = torch.zeros(1, self.batch_size, self.film_fc6_dim, device=self.device)
                
    ######################################
    ### Initialize Kalman Gain Network ###
    ######################################
    def InitKGainNet(self, prior_Q, prior_Sigma, prior_S, args):

        self.seq_len_input = 1 # KNet calculates time-step by time-step
        self.batch_size = args.n_batch # Batch size

        self.prior_Q = prior_Q.to(self.device)
        self.prior_Sigma = prior_Sigma.to(self.device)
        self.prior_S = prior_S.to(self.device)

        # GRU to track Q
        self.d_input_Q = self.m * args.in_mult_KNet
        self.d_hidden_Q = self.m ** 2
        self.GRU_Q = nn.GRU(self.d_input_Q, self.d_hidden_Q).to(self.device)

        # GRU to track Sigma
        self.d_input_Sigma = self.d_hidden_Q + self.m * args.in_mult_KNet
        self.d_hidden_Sigma = self.m ** 2
        self.GRU_Sigma = nn.GRU(self.d_input_Sigma, self.d_hidden_Sigma).to(self.device)
       
        # GRU to track S
        self.d_input_S = self.n ** 2 + 2 * self.n * args.in_mult_KNet
        self.d_hidden_S = self.n ** 2
        self.GRU_S = nn.GRU(self.d_input_S, self.d_hidden_S).to(self.device)
        
        # Fully connected 1
        self.d_input_FC1 = self.d_hidden_Sigma
        self.d_output_FC1 = self.n ** 2
        self.FC1 = nn.Sequential(
                nn.Linear(self.d_input_FC1, self.d_output_FC1),
                nn.ReLU()).to(self.device)

        # Fully connected 2
        self.d_input_FC2 = self.d_hidden_S + self.d_hidden_Sigma
        self.d_output_FC2 = self.n * self.m
        self.d_hidden_FC2 = self.d_input_FC2 * args.out_mult_KNet
        self.FC2 = nn.Sequential(
                nn.Linear(self.d_input_FC2, self.d_hidden_FC2),
                nn.ReLU(),
                nn.Linear(self.d_hidden_FC2, self.d_output_FC2)).to(self.device)
        
        # Initialize final layer weights to be very small
        # This ensures K starts near 0, making the filter initially stable (pure prediction)        
        with torch.no_grad():
            self.FC2[-1].weight.mul_(0.001)
            self.FC2[-1].bias.zero_()

        # Fully connected 3
        self.d_input_FC3 = self.d_hidden_S + self.d_output_FC2
        self.d_output_FC3 = self.m ** 2
        self.FC3 = nn.Sequential(
                nn.Linear(self.d_input_FC3, self.d_output_FC3),
                nn.ReLU()).to(self.device)

        # Fully connected 4
        self.d_input_FC4 = self.d_hidden_Sigma + self.d_output_FC3
        self.d_output_FC4 = self.d_hidden_Sigma
        self.FC4 = nn.Sequential(
                nn.Linear(self.d_input_FC4, self.d_output_FC4),
                nn.ReLU()).to(self.device)
        
        # Fully connected 5
        self.d_input_FC5 = self.m
        self.d_output_FC5 = self.m * args.in_mult_KNet
        self.FC5 = nn.Sequential(
                nn.Linear(self.d_input_FC5, self.d_output_FC5),
                nn.ReLU()).to(self.device)

        # Fully connected 6
        self.d_input_FC6 = self.m
        self.d_output_FC6 = self.m * args.in_mult_KNet
        self.FC6 = nn.Sequential(
                nn.Linear(self.d_input_FC6, self.d_output_FC6),
                nn.ReLU()).to(self.device)
        
        # Fully connected 7
        self.d_input_FC7 = 2 * self.n
        self.d_output_FC7 = 2 * self.n * args.in_mult_KNet
        self.FC7 = nn.Sequential(
                nn.Linear(self.d_input_FC7, self.d_output_FC7),
                nn.ReLU()).to(self.device)        

    ##################################
    ### Initialize System Dynamics ###
    ##################################
    def InitSystemDynamics(self, f, h, m, n):
        
        # Set State Evolution Function
        self.f = f
        self.m = m

        # Set Observation Function
        self.h = h
        self.n = n

    ###########################
    ### Initialize Sequence ###
    ###########################
    def InitSequence(self, M1_0, T):
        """
        input M1_0 (torch.tensor): 1st moment of x at time 0 [batch_size, m, 1]
        """
        self.T = T

        self.m1x_posterior = self._sanitize(M1_0.to(self.device), clamp_value=self.state_clip_value)
        self.m1x_posterior_previous = self.m1x_posterior
        self.m1x_prior_previous = self.m1x_posterior
        self.y_previous = self._sanitize(
            self.h(self.m1x_posterior),
            clamp_value=self.measurement_clip_value,
        )

    ######################
    ### Compute Priors ###
    ######################
    def step_prior(self):
        # Predict the 1-st moment of x
        self.m1x_prior = self._sanitize(
            self.f(self.m1x_posterior),
            clamp_value=self.state_clip_value,
        )

        # Predict the 1-st moment of y
        self.m1y = self._sanitize(
            self.h(self.m1x_prior),
            clamp_value=self.measurement_clip_value,
        )

    ##############################
    ### Kalman Gain Estimation ###
    ##############################
    def step_KGain_est(self, y):
        y = self._sanitize(y, clamp_value=self.measurement_clip_value)
        # both in size [batch_size, n]
        obs_diff = torch.squeeze(y,2) - torch.squeeze(self.y_previous,2) 
        obs_innov_diff = torch.squeeze(y,2) - torch.squeeze(self.m1y,2)
        if self.meas_type == "RANGE_BEARING":
            # Optionally wrap bearing residuals (rad)
            if self.rb_wrap_theta:
                obs_diff[:, 1] = wrap_to_pi(obs_diff[:, 1])
                obs_innov_diff[:, 1] = wrap_to_pi(obs_innov_diff[:, 1])

            # Optionally convert bearing residual (rad) -> tangential displacement (m)
            if self.rb_arc_length:
                # Cap the range factor in the normalized benchmark regime so the
                # tangentialized bearing residual does not dominate the network
                # features at long sequence lengths.
                r_scale = torch.clamp(torch.squeeze(y, 2)[:, 0], min=1.0, max=5.0)  # [B]
                obs_diff[:, 1] = obs_diff[:, 1] * r_scale
                obs_innov_diff[:, 1] = obs_innov_diff[:, 1] * r_scale
        
        
        # both in size [batch_size, m]
        fw_evol_diff = torch.squeeze(self.m1x_posterior,2) - torch.squeeze(self.m1x_posterior_previous,2)
        fw_update_diff = torch.squeeze(self.m1x_posterior,2) - torch.squeeze(self.m1x_prior_previous,2)
        
        obs_diff = self._safe_normalize(obs_diff)
        obs_innov_diff = self._safe_normalize(obs_innov_diff)
        fw_evol_diff = self._safe_normalize(fw_evol_diff)
        fw_update_diff = self._safe_normalize(fw_update_diff)
        

        # Kalman Gain Network Step
        KG = self.KGain_step(obs_diff, obs_innov_diff, fw_evol_diff, fw_update_diff)

        # Reshape Kalman Gain to a Matrix
        self.KGain = self._sanitize(
            torch.reshape(KG, (self.batch_size, self.m, self.n)),
            clamp_value=self.gain_clip_value,
        )

    #######################
    ### Kalman Net Step ###
    #######################
    def KNet_step(self, y):
        y = self._sanitize(y, clamp_value=self.measurement_clip_value)

        # Compute Priors
        self.step_prior()

        # Compute Kalman Gain
        self.step_KGain_est(y)

        # Innovation
        dy = y - self.m1y  # [batch_size, n, 1]
        if self.meas_type == "RANGE_BEARING":
            # Optionally wrap bearing residuals (rad)
            if self.rb_wrap_theta:
                dy[:, 1, 0] = wrap_to_pi(dy[:, 1, 0])

            # Optionally convert bearing residual (rad) -> tangential displacement (m)
            if self.rb_arc_length:
                # Match the cap used in step_KGain_est so feature scaling stays
                # consistent between gain estimation and state correction.
                r_scale = torch.clamp(y[:, 0, 0], min=1.0, max=5.0)
                dy[:, 1, 0] = dy[:, 1, 0] * r_scale

        # Safety clamp: prevents one extreme innovation component from causing
        # a catastrophic state update in the long-context setting.
        dy = self._sanitize(torch.clamp(dy, -10.0, 10.0), clamp_value=10.0)

        # Compute the 1-st posterior moment
        INOV = self._sanitize(
            torch.bmm(self.KGain, dy),
            clamp_value=self.state_clip_value,
        )
        self.m1x_posterior_previous = self.m1x_posterior
        self.m1x_posterior = self._sanitize(
            self.m1x_prior + INOV,
            clamp_value=self.state_clip_value,
        )

        epsilon = y - self._sanitize(
            self.h(self.m1x_posterior),
            clamp_value=self.measurement_clip_value,
        )
        if self.meas_type == "RANGE_BEARING":
            # Optionally wrap bearing residuals (rad)
            if self.rb_wrap_theta:
                epsilon[:, 1, 0] = wrap_to_pi(epsilon[:, 1, 0])

            # Optionally convert bearing residual (rad) -> tangential displacement (m)
            if self.rb_arc_length:
                r_scale = torch.clamp(y[:, 0, 0], min=1.0, max=5.0)
                epsilon[:, 1, 0] = epsilon[:, 1, 0] * r_scale


        #self.state_process_posterior_0 = self.state_process_prior_0
        self.m1x_prior_previous = self.m1x_prior

        # update y_prev
        self.y_previous = self._sanitize(y, clamp_value=self.measurement_clip_value)
        
        # Scalars for controller (scale cues)
        dy_mag = torch.log(torch.norm(dy.squeeze(2), dim=1, p=2) + 1e-12)        # [B]
        eps_mag = torch.log(torch.norm(epsilon.squeeze(2), dim=1, p=2) + 1e-12)  # [B]

        dy_mag = self._sanitize(dy_mag.detach(), clamp_value=20.0)
        eps_mag = self._sanitize(eps_mag.detach(), clamp_value=20.0)
        
        # return
        return self.m1x_posterior, dy_mag, eps_mag, self.KGain

    ########################
    ### Kalman Gain Step ###
    ########################
    def KGain_step(self, obs_diff, obs_innov_diff, fw_evol_diff, fw_update_diff):

        def expand_dim(x):
            expanded = torch.empty(self.seq_len_input, self.batch_size, x.shape[-1]).to(self.device)
            expanded[0, :, :] = x
            return expanded

        obs_diff = expand_dim(obs_diff)
        obs_innov_diff = expand_dim(obs_innov_diff)
        fw_evol_diff = expand_dim(fw_evol_diff)
        fw_update_diff = expand_dim(fw_update_diff)

        ####################
        ### Forward Flow ###
        ####################
        
        # FC 5
        in_FC5 = fw_update_diff
        out_FC5 = self._sanitize(self.FC5(in_FC5), clamp_value=20.0)
        
        ######## FiLM layer FC5 #################
        if self.film is True:
            out_FC5 = self.apply_film(out_FC5, self.gamma_fc5, self.beta_fc5)
        
        # Q-GRU
        in_Q = out_FC5
        out_Q, self.h_Q = self.GRU_Q(in_Q, self.h_Q)
        out_Q = self._sanitize_hidden(out_Q)
        self.h_Q = self._sanitize_hidden(self.h_Q)

        # FC 6
        in_FC6 = fw_evol_diff
        out_FC6 = self._sanitize(self.FC6(in_FC6), clamp_value=20.0)
        
        ######## FiLM layer FC6 #################
        if self.film is True:
            out_FC6 = self.apply_film(out_FC6, self.gamma_fc6, self.beta_fc6)
        
        # Sigma_GRU
        in_Sigma = torch.cat((out_Q, out_FC6), 2)
        out_Sigma, self.h_Sigma = self.GRU_Sigma(in_Sigma, self.h_Sigma)
        out_Sigma = self._sanitize_hidden(out_Sigma)
        self.h_Sigma = self._sanitize_hidden(self.h_Sigma)

        # FC 1
        in_FC1 = out_Sigma
        out_FC1 = self._sanitize(self.FC1(in_FC1), clamp_value=20.0)

        # FC 7
        in_FC7 = torch.cat((obs_diff, obs_innov_diff), 2)
        out_FC7 = self._sanitize(self.FC7(in_FC7), clamp_value=20.0)
        
        ######## FiLM layer FC7 #################
        if self.film is True:
            out_FC7 = self.apply_film(out_FC7, self.gamma_fc7, self.beta_fc7)


        # S-GRU
        in_S = torch.cat((out_FC1, out_FC7), 2)
        out_S, self.h_S = self.GRU_S(in_S, self.h_S)
        out_S = self._sanitize_hidden(out_S)
        self.h_S = self._sanitize_hidden(self.h_S)
        
        #Added to tap S matrix
        self.out_S = out_S
        
        # FC 2
        in_FC2 = torch.cat((out_Sigma, out_S), 2)
        out_FC2 = self._sanitize(self.FC2(in_FC2), clamp_value=self.gain_clip_value)

        #####################
        ### Backward Flow ###
        #####################

        # FC 3
        in_FC3 = torch.cat((out_S, out_FC2), 2)
        out_FC3 = self._sanitize(self.FC3(in_FC3), clamp_value=20.0)

        # FC 4
        in_FC4 = torch.cat((out_Sigma, out_FC3), 2)
        out_FC4 = self._sanitize(self.FC4(in_FC4), clamp_value=20.0)

        # updating hidden state of the Sigma-GRU
        self.h_Sigma = self._sanitize_hidden(out_FC4)

        return out_FC2
    ###############
    ### Forward ###
    ###############
    def forward(self, y):
        y = self._sanitize(y.to(self.device), clamp_value=self.measurement_clip_value)
        return self.KNet_step(y)

    #########################
    ### Init Hidden State ###
    #########################
    def init_hidden_KNet(self):
        weight = next(self.parameters()).data
        hidden = weight.new(self.seq_len_input, self.batch_size, self.d_hidden_S).zero_()
        self.h_S = hidden.data
        self.h_S = self.prior_S.flatten().reshape(1, 1, -1).repeat(self.seq_len_input,self.batch_size, 1) # batch size expansion
        hidden = weight.new(self.seq_len_input, self.batch_size, self.d_hidden_Sigma).zero_()
        self.h_Sigma = hidden.data
        self.h_Sigma = self.prior_Sigma.flatten().reshape(1,1, -1).repeat(self.seq_len_input,self.batch_size, 1) # batch size expansion
        hidden = weight.new(self.seq_len_input, self.batch_size, self.d_hidden_Q).zero_()
        self.h_Q = hidden.data
        self.h_Q = self.prior_Q.flatten().reshape(1,1, -1).repeat(self.seq_len_input,self.batch_size, 1) # batch size expansion

    def save_internal_state(self):
        self._saved_state = {
            'm1x_posterior': self.m1x_posterior.clone(),
            'm1x_posterior_previous': self.m1x_posterior_previous.clone(),
            'm1x_prior_previous': self.m1x_prior_previous.clone(),
            'y_previous': self.y_previous.clone(),
            'h_S': self.h_S.clone(),
            'h_Sigma': self.h_Sigma.clone(),
            'h_Q': self.h_Q.clone(),
        }
    
    def restore_internal_state(self):
        state = self._saved_state
        self.m1x_posterior = state['m1x_posterior']
        self.m1x_posterior_previous = state['m1x_posterior_previous']
        self.m1x_prior_previous = state['m1x_prior_previous']
        self.y_previous = state['y_previous']
        self.h_S = state['h_S']
        self.h_Sigma = state['h_Sigma']
        self.h_Q = state['h_Q']



### knet_nn_arch1.py

**File:** `knet_nn_arch1.py`


In [ ]:
####################################
'''
KalmanNet Architecture 1 (Single GRU)
'''
####################################

import torch
import torch.nn as nn
import torch.nn.functional as func
import math

TWO_PI = 2.0 * math.pi

def wrap_to_pi(angle):
    return (angle + math.pi) % (2.0 * math.pi) - math.pi

class KalmanNetNN(torch.nn.Module):

    ###################
    ### Constructor ###
    ###################
    def __init__(self):
        super().__init__()

    def NNBuild(self, SysModel, args):
        # Device
        if args.use_cuda:
            self.device = torch.device('cuda')
        else:
            self.device = torch.device('cpu')

        self.InitSystemDynamics(SysModel.f, SysModel.h, SysModel.m, SysModel.n)
        self.InitKGainNet(args)
        
        self.meas_type = args.meas_type

        # RANGE_BEARING residual handling
        self.rb_wrap_theta = True
        self.rb_arc_length = True

    ######################################
    ### Initialize Kalman Gain Network ###
    ######################################
    def InitKGainNet(self, args):
        self.seq_len_input = 1
        self.batch_size = args.n_batch

        # Input dimensions based on the 4 features: 
        # 1. obs_diff (n)
        # 2. obs_innov_diff (n)
        # 3. fw_evol_diff (m)
        # 4. fw_update_diff (m)
        self.d_input = (self.n * 2) + (self.m * 2)
        
        # Dimensions for internal layers
        self.d_fc1 = (self.m + self.n) * 10 * 8 
        
        self.d_fc2 = (self.m * self.n) * 1 * 4
        
        # Output is the flattened Kalman Gain matrix (m * n)
        self.d_output = self.m * self.n

        # GRU Hidden Size
        # Paper suggests hidden state size related to m^2 + n^2
        scale_factor = 10
        self.d_hidden_gru = scale_factor * (self.m**2 + self.n**2)

        # Architecture 1 Structure
        
        # Fully Connected Input Layer
        self.FC_Input = nn.Sequential(
            nn.Linear(self.d_input, self.d_fc1),
            nn.ReLU()
        ).to(self.device)

        # Single GRU Layer
        self.GRU = nn.GRU(self.d_fc1, self.d_hidden_gru).to(self.device)

        # Fully Connected Output Layer
        self.FC_Output = nn.Sequential(
            nn.Linear(self.d_hidden_gru, self.d_fc2),
            nn.ReLU(),
            nn.Linear(self.d_fc2, self.d_output)
        ).to(self.device)
        
        '''
        Initialize final layer weights to be very small
        This ensures K starts near 0, making the filter initially stable
        '''
        with torch.no_grad():
            self.FC_Output[-1].weight.mul_(0.001)
            self.FC_Output[-1].bias.zero_()
        


    ##################################
    ### Initialize System Dynamics ###
    ##################################
    def InitSystemDynamics(self, f, h, m, n):
        self.f = f
        self.m = m
        self.h = h
        self.n = n

    ###########################
    ### Initialize Sequence ###
    ###########################
    def InitSequence(self, M1_0, T):
        self.T = T
        self.m1x_posterior = M1_0.to(self.device)
        self.m1x_posterior_previous = self.m1x_posterior
        self.m1x_prior_previous = self.m1x_posterior
        self.y_previous = self.h(self.m1x_posterior)

    ######################
    ### Compute Priors ###
    ######################
    def step_prior(self):
        self.m1x_prior = self.f(self.m1x_posterior)
        self.m1y = self.h(self.m1x_prior)

    ##############################
    ### Kalman Gain Estimation ###
    ##############################
    def step_KGain_est(self, y):
        # Observation Difference (y_t - y_{t-1}) [batch, n]
        obs_diff = torch.squeeze(y, 2) - torch.squeeze(self.y_previous, 2)
        
        # Innovation (y_t - y_{t|t-1}) [batch, n]
        obs_innov_diff = torch.squeeze(y, 2) - torch.squeeze(self.m1y, 2)

        # Handle Range Bearing wrap/scaling logic
        if self.meas_type == "RANGE_BEARING":
            if self.rb_wrap_theta:
                obs_diff[:, 1] = wrap_to_pi(obs_diff[:, 1])
                obs_innov_diff[:, 1] = wrap_to_pi(obs_innov_diff[:, 1])
            if self.rb_arc_length:
                r_scale = torch.clamp(torch.squeeze(y, 2)[:, 0], min=1.0)
                obs_diff[:, 1] = obs_diff[:, 1] * r_scale
                obs_innov_diff[:, 1] = obs_innov_diff[:, 1] * r_scale

        # Forward Evolution Diff (x_{t|t} - x_{t-1|t-1}) [batch, m]
        fw_evol_diff = torch.squeeze(self.m1x_posterior, 2) - torch.squeeze(self.m1x_posterior_previous, 2)
        
        # Forward Update Diff (x_{t|t} - x_{t|t-1}) [batch, m]
        fw_update_diff = torch.squeeze(self.m1x_posterior, 2) - torch.squeeze(self.m1x_prior_previous, 2)

        # Normalize features
        obs_diff = func.normalize(obs_diff, p=2, dim=1, eps=1e-12)
        obs_innov_diff = func.normalize(obs_innov_diff, p=2, dim=1, eps=1e-12)
        fw_evol_diff = func.normalize(fw_evol_diff, p=2, dim=1, eps=1e-12)
        fw_update_diff = func.normalize(fw_update_diff, p=2, dim=1, eps=1e-12)

        # Forward Pass
        
        # [batch, 2n + 2m]
        in_features = torch.cat((obs_diff, obs_innov_diff, fw_evol_diff, fw_update_diff), dim=1)
        
        in_features = in_features.unsqueeze(0)

        # FC Input
        out_fc1 = self.FC_Input(in_features)

        # GRU
        out_gru, self.h_gru = self.GRU(out_fc1, self.h_gru)

        # FC Output
        out_fc_final = self.FC_Output(out_gru)

        # Shape the output to Kalman Gain Matrix [batch, m, n]
        KG_flat = out_fc_final[0] 
        self.KGain = torch.reshape(KG_flat, (self.batch_size, self.m, self.n))

    #######################
    ### Kalman Net Step ###
    #######################
    def KNet_step(self, y):
        # Compute Priors
        self.step_prior()

        # Compute Kalman Gain 
        self.step_KGain_est(y)

        # Innovation
        dy = y - self.m1y
        
        if self.meas_type == "RANGE_BEARING":
            if self.rb_wrap_theta:
                dy[:, 1, 0] = wrap_to_pi(dy[:, 1, 0])
            if self.rb_arc_length:
                r_scale = torch.clamp(y[:, 0, 0], min=1.0)
                dy[:, 1, 0] = dy[:, 1, 0] * r_scale

        # State Update 
        INOV = torch.bmm(self.KGain, dy)
        self.m1x_posterior_previous = self.m1x_posterior
        self.m1x_posterior = self.m1x_prior + INOV

        # Compute Error 
        epsilon = y - self.h(self.m1x_posterior)
        
        self.m1x_prior_previous = self.m1x_prior
        self.y_previous = y
        
        # Scalars for controller (monitoring)
        dy_mag = torch.log(torch.norm(dy.squeeze(2), dim=1, p=2) + 1e-12)        # [B]
        eps_mag = torch.log(torch.norm(epsilon.squeeze(2), dim=1, p=2) + 1e-12)  # [B]

        dy_mag = dy_mag.detach()
        eps_mag = eps_mag.detach()
        
        return self.m1x_posterior, dy_mag, eps_mag, self.KGain

    ###############
    ### Forward ###
    ###############
    def forward(self, y):
        y = y.to(self.device)
        return self.KNet_step(y)

    #########################
    ### Init Hidden State ###
    #########################
    def init_hidden_KNet(self):
        # Initialize the single GRU hidden state
        weight = next(self.parameters()).data
        hidden = weight.new(self.seq_len_input, self.batch_size, self.d_hidden_gru).zero_()
        self.h_gru = hidden.data

    def save_internal_state(self):
        self._saved_state = {
            'm1x_posterior': self.m1x_posterior.clone(),
            'm1x_posterior_previous': self.m1x_posterior_previous.clone(),
            'm1x_prior_previous': self.m1x_prior_previous.clone(),
            'y_previous': self.y_previous.clone(),
            'h_gru': self.h_gru.clone(),
        }
    
    def restore_internal_state(self):
        state = self._saved_state
        self.m1x_posterior = state['m1x_posterior']
        self.m1x_posterior_previous = state['m1x_posterior_previous']
        self.m1x_prior_previous = state['m1x_prior_previous']
        self.y_previous = state['y_previous']
        self.h_gru = state['h_gru']


### train_validate_test.py

**File:** `train_validate_test.py`


In [ ]:
import numpy as np
import torch
import copy


def _grads_are_finite(model):
    for param in model.parameters():
        if param.grad is not None and not torch.isfinite(param.grad).all():
            return False
    return True


def _tensor_is_finite(value):
    return torch.isfinite(value).all().item()


def evaluate_loss_knet(kalmannet, x_pool, y_pool, init_val, args, loss_fn=torch.nn.MSELoss(reduction='mean')):
    # Evaluate loss on a dataset
    kalmannet.eval()
    kalmannet.film = False
    eval_seq_len = x_pool.shape[2]
    Kfro_list = []
    log_norm_innov_list = []
    
    with torch.no_grad():
        kalmannet.batch_size = x_pool.shape[0]
        kalmannet.init_hidden_KNet()
        kalmannet.InitSequence(init_val, eval_seq_len)
        x_est = torch.zeros_like(x_pool)
        
        # Iterate through time steps (Validation uses full sequence length)
        for tidx in range(eval_seq_len):
            temp_in = y_pool[:, :, tidx].unsqueeze(2)
            temp, log_norm_innov, _, K_t = kalmannet(temp_in)
            x_est[:, :, tidx] = temp.squeeze(-1)
            
            # Frobenius norm, averaged over batch, for debug
            Kfro = torch.norm(K_t.reshape(K_t.shape[0], -1), dim=1)  # [B]
            Kfro_list.append(Kfro.mean().item())
            log_norm_innov_list.append(log_norm_innov.mean().item())
        
        # Calculate validation loss on the full sequence
        loss = loss_fn(x_est, x_pool)
        
        return loss.item(), x_est, Kfro_list, log_norm_innov_list

def train_validate_knet(model, kalmannet, args, x_train, y_train, x_val, y_val, init_train, init_val, filename):
 
    # Long-context KalmanNet training is substantially less stable than the
    # original short-sequence setup, so we use a lower effective LR once the
    # sequence length moves beyond the reference regime.
    if args.seq_len > 100:
        seq_ratio = max(float(args.seq_len) / 100.0, 1.0)
        long_seq_lr = args.learning_rate / (seq_ratio ** 2)
        long_seq_lr = max(long_seq_lr, 1e-5)
    else:
        long_seq_lr = args.learning_rate

    optimizer = torch.optim.Adam(kalmannet.parameters(),
                                 lr=long_seq_lr,
                                 weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=8,
        min_lr=1e-7
    )
    loss_fn = torch.nn.MSELoss(reduction='mean')
    
    # Early stopping params
    patience = args.stopping_criteria 
    min_delta_db = args.update_thr_db 

    best_val_loss = float('inf')
    best_epoch = -1
    best_model = None

    no_improve_counter = 0
    dataset_size = x_train.shape[0]
    n_batches = max(1, dataset_size // args.n_batch)
    val_loss_traindB = []
    
    # ------------------------------------------------------------------------
    # TBPTT Parameters 
    # k: Detach gradients every k steps (prevents explosion)
    # w: Update weights every w steps (sliding window)
    # Recommended values from paper: k=2, w=4 (or w=8)
    # ------------------------------------------------------------------------
    k_steps = 2
    w_steps = 4 
    if args.seq_len > 200:
        k_steps = 1
        w_steps = 2

    for epoch in range(args.epochs):
        kalmannet.train()
        kalmannet.film = False
        permutation = torch.randperm(dataset_size)
        epoch_loss = 0.0
        batches_updated = 0 

        nan_steps_count = 0 
        
        # Full pass through dataset
        for i in range(0, dataset_size, args.n_batch):
            idxs = permutation[i:i + args.n_batch]
            ts_batch, obs_batch, init_batch = x_train[idxs], y_train[idxs], init_train[idxs]

            kalmannet.batch_size = ts_batch.shape[0]
            kalmannet.init_hidden_KNet()
            kalmannet.InitSequence(init_batch, args.seq_len)

            optimizer.zero_grad()
            
            # Accumulators for the sliding window
            current_window_loss = 0.0
            window_steps = 0
            batch_total_loss = 0.0
            
            debug_vec = torch.zeros_like(ts_batch)
            
            # Iterate over sequence length (Time T)
            seq_len = ts_batch.shape[2]
            
            for tidx in range(seq_len):
                # Forward Step
                # Input: [Batch, Dim, 1]
                kalmannet.save_internal_state()
                temp, _, _, _ = kalmannet(obs_batch[:, :, tidx].unsqueeze(2))
                if (
                    not _tensor_is_finite(temp)
                    or not _tensor_is_finite(kalmannet.m1x_posterior)
                    or not _tensor_is_finite(kalmannet.KGain)
                ):
                    kalmannet.restore_internal_state()
                    nan_steps_count += 1
                    continue
                
                target = ts_batch[:, :, tidx]
                
                # Calculate step loss
                step_loss = loss_fn(temp.squeeze(-1), target)
                
                if not torch.isfinite(step_loss):
                    kalmannet.restore_internal_state()
                    nan_steps_count += 1
                    continue 

                current_window_loss += step_loss
                window_steps += 1
                
                # TBPTT: Detach Gradients (k)
                # Detach hidden state every k steps
                # tidx is 0 indexed, so check (tidx + 1)
                if (tidx + 1) % k_steps == 0:
                    # Architecture 2 Hidden States
                    if hasattr(kalmannet, 'h_S'):
                        kalmannet.h_S = kalmannet.h_S.detach()
                    if hasattr(kalmannet, 'h_Sigma'):
                        kalmannet.h_Sigma = kalmannet.h_Sigma.detach()
                    if hasattr(kalmannet, 'h_Q'):
                        kalmannet.h_Q = kalmannet.h_Q.detach()
                    
                    # Architecture 1 Hidden State
                    if hasattr(kalmannet, 'h_gru'):
                        kalmannet.h_gru = kalmannet.h_gru.detach()
                        
                    if hasattr(kalmannet, 'm1x_posterior'):
                        kalmannet.m1x_posterior = kalmannet.m1x_posterior.detach()
                    if hasattr(kalmannet, 'm1x_posterior_previous'):
                        kalmannet.m1x_posterior_previous = kalmannet.m1x_posterior_previous.detach()
                    if hasattr(kalmannet, 'm1x_prior_previous'):
                        kalmannet.m1x_prior_previous = kalmannet.m1x_prior_previous.detach()
                    if hasattr(kalmannet, 'y_previous'):
                        kalmannet.y_previous = kalmannet.y_previous.detach()

                # Update weights every w steps OR at the end of the sequence
                if (tidx + 1) % w_steps == 0 or (tidx + 1) == seq_len:
                    
                    if window_steps > 0:
                        # Average loss over the window
                        loss_to_backprop = current_window_loss / window_steps
                        
                        if not torch.isfinite(loss_to_backprop) or loss_to_backprop.item() > 1e6:
                            optimizer.zero_grad()
                            nan_steps_count += window_steps
                            current_window_loss = 0.0
                            window_steps = 0
                            continue

                        loss_to_backprop.backward()

                        if not _grads_are_finite(kalmannet):
                            optimizer.zero_grad()
                            nan_steps_count += window_steps
                            current_window_loss = 0.0
                            window_steps = 0
                            continue
                        
                        # Gradient Clipping
                        torch.nn.utils.clip_grad_norm_(kalmannet.parameters(), max_norm=0.1)
                        
                        optimizer.step()
                        optimizer.zero_grad()
                        
                        # total batch loss for info only
                        batch_total_loss += loss_to_backprop.item()
                    
                    # Reset
                    current_window_loss = 0.0
                    window_steps = 0
            
            # End of batch
            if seq_len > 0:
                epoch_loss += batch_total_loss / (seq_len / w_steps) # Approx average
                batches_updated += 1
            else:
                print(f"  Epoch {epoch+1} Batch {i//args.n_batch}: All steps NaN or empty.")

        # Compute average epoch loss
        if batches_updated > 0:
            epoch_loss /= batches_updated
        else:
            epoch_loss = float('nan') 

        # Validation (Always on full sequence length, no truncation)
        val_loss, _, _, _ = evaluate_loss_knet(kalmannet, x_val, y_val, init_val, args, loss_fn)
        if np.isfinite(val_loss):
            scheduler.step(val_loss)
        
        train_db = 10*np.log10(epoch_loss) if epoch_loss > 0 else -100
        val_db = 10*np.log10(val_loss) if np.isfinite(val_loss) and val_loss > 0 else float('nan')
        
        print(f"Epoch {epoch+1}, Train Loss(dB): {train_db:.6f}, Val Loss(dB): {val_db:.6f}")
        
        if nan_steps_count > 0:
            print(f"  Epoch {epoch+1}: Found {nan_steps_count} NaN steps.")

        val_loss_traindB.append(val_db)
        
        # Improvement check
        current_best_db = 10*np.log10(best_val_loss) if best_val_loss != float('inf') else float('inf')
        
        if val_db < current_best_db - min_delta_db:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_model = copy.deepcopy(kalmannet.state_dict())
            torch.save(best_model, filename)
            no_improve_counter = 0
        else:
            no_improve_counter += 1

        # Early stopping condition
        if no_improve_counter >= patience:
            print(f"\nEarly stopping triggered after {patience} epochs without validation improvement.")
            break

    print(f"\nBest Validation Loss(dB): {10*np.log10(best_val_loss):.6f} at Epoch {best_epoch}")
    if best_model is not None:
        kalmannet.load_state_dict(best_model)
        print(f"Best model loaded from memory and saved to {filename}.")
    else:
        print("Warning: no improvement observed, final weights used.")

    return kalmannet, val_loss_traindB


### main.py

**File:** `main.py`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch


from sim_params import simparams_class, validate_simparams

from data_model import ConstantAcceleration2DModel

from data_generation import generate_synthetic_data, split_data

from ekf import extended_kf, batch_ekf, compute_lossvstime

from knet_nn import KalmanNetNN

#from knet_nn_arch1 import KalmanNetNN

from train_validate_test import train_validate_knet, evaluate_loss_knet


if __name__ == "__main__":
    
    print("\nExecuting main...")

    simparams = simparams_class()

    validate_simparams(simparams)
    
    ca_model = ConstantAcceleration2DModel(simparams)
    
    ###Data generation####
    
    truestate_dataset_20snr, observation_dataset_20snr, init_state_20snr, Q_20snr, R_20snr = generate_synthetic_data(ca_model, simparams, snr_dB = 20)
    
    # SCALE CORRECTION: m -> km
    SCALE = 1000.0 
    
    # Scale True State (x, vx, ax, y, vy, ay)
    truestate_dataset_20snr /= SCALE
    
    # Scale Initialization (Same as state)
    init_state_20snr /= SCALE
    
    # Scale Process Noise Q (Variance of state)
    Q_20snr /= (SCALE ** 2)
    
    # Scale Observations only range
    observation_dataset_20snr[:, 0, :] /= SCALE
    
    # Scale Measurement Noise R
    R_20snr[0, 0] /= (SCALE ** 2)
     
    # Split into train and validation sets
    (ts_train_all, obs_train_all, init_train_all), (ts_val_all, obs_val_all, init_val_all) = split_data(truestate_dataset_20snr, observation_dataset_20snr, init_state_20snr)

    
    ######Train###########################
    kalmannet = KalmanNetNN()
    kalmannet.NNBuild(ca_model, simparams)    
    
    if(simparams.train_knet == True):
        print("\nStarting KalmanNet training....") 
        kalmannet,val_loss_train = train_validate_knet(ca_model, kalmannet, simparams, ts_train_all, obs_train_all, ts_val_all, obs_val_all, init_train_all, init_val_all, simparams.knet_name)
    else:
        print("Loading saved KalmanNet model...") 
        state_dict = torch.load(simparams.knet_name, weights_only=True)
        kalmannet.load_state_dict(state_dict)
        
    
    ######Evaluate###########################
    ts_test, obs_test, init_test, Q_test, R_test = generate_synthetic_data(ca_model, simparams, snr_dB = 20, seq_mul_fact=1)
        
    # Scale test data also
    ts_test /= SCALE
    init_test /= SCALE
    Q_test /= (SCALE ** 2)
    obs_test[:, 0, :] /= SCALE  # Scale Range
    R_test[0, 0] /= (SCALE ** 2) # Scale Range Variance
    
    ekf = extended_kf(ca_model, Q_test/simparams.q2_scale, R_test/simparams.r2_scale, simparams)
    batch_ekf_eval = batch_ekf(ekf)
    ekf_estimates, P_seq, K_seq, S_seq, NU_seq = batch_ekf_eval.evaluate_batch(ts_test, obs_test, init_test)
    ekf_loss_time, ekf_tot_lossdB = compute_lossvstime(ts_test, ekf_estimates)
        
    temp_loss, x_est, K_fro, ln_innov = evaluate_loss_knet(kalmannet, ts_test, obs_test, init_test, simparams)
    knet_loss_time, knet_tot_lossdB = compute_lossvstime(ts_test, x_est)
    
    print(f"Loss KNet: {knet_tot_lossdB:.3f} \t Loss EKF: {ekf_tot_lossdB:.3f}")
    
    time_idx = list(range(0, x_est.shape[2]))

    plt.figure(figsize=(8, 5))
    plt.plot(time_idx, knet_loss_time, marker='o', label="KNet loss")
    plt.plot(time_idx, ekf_loss_time, marker='s', label="EKF loss")
    
    plt.xlabel("Time idx")
    plt.ylabel("MSE (dB)")
    plt.title("MSE vs Time")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig("MSE_Time.pdf",  bbox_inches="tight")
    plt.show()
    
    plt.figure()
    plt.plot(K_fro, marker='o')
    plt.title("KNet Kalman gain: mean Frobenius ||K_t||_F vs time")
    plt.xlabel("t")
    plt.ylabel("mean ||K||_F")
    plt.grid(True)
    
    plt.figure()
    plt.plot(ln_innov, marker='.')
    plt.title("Log Norm of Innovation vs time")
    plt.xlabel("t")
    plt.ylabel("log norm of innovation")
    plt.grid(True)
    
    plt.show()
    
    if(simparams.train_knet == "True"):
        plt.figure()
        iterations_train = list(range(len(val_loss_train)))
        plt.plot(iterations_train, val_loss_train, marker="*")
        plt.xlabel('iteration')
        plt.ylabel('Validation loss (dB)')
        plt.title('Validation loss w.r.t time')
        plt.grid(True)
        plt.savefig("validation_curve.pdf", bbox_inches="tight")
        plt.show()
    
 
    
    print("\n##################End of program#####################\n\n\n")
    
    


 


## BiMamba Benchmark

The active neural tracker is not a local `mamba_tracker.py` file anymore. The benchmark now uses the shared BiMamba model from the `BiMambaTracker` package.

Key current details:
- the benchmark script imports `BiMambaTrackerModel`
- the temporal encoder lives in `bimamba_encoder.py`
- the residual output head lives in `prediction_head.py`
- KalmanNet checkpoints are evaluated and the best validation checkpoint is selected automatically
- `seq_len = 100` is the main benchmark context used by default


### ../../BiMambaTracker/models/tracker_model.py


In [ ]:
import math

import torch
import torch.nn as nn

from models.bimamba_encoder import BiMambaEncoder
from models.prediction_head import PredictionHead


def wrap_to_pi(angle):

    return (angle + math.pi) % (2.0 * math.pi) - math.pi


class BiMambaTrackerModel(nn.Module):

    def __init__(
        self,
        state_dim=6,
        obs_dim=2,
        dt=1.0,
        embed_dim=128,
        hidden_dim=256,
        num_layers=3,
        d_state=16,
        d_conv=4,
        expand=2,
        dropout=0.08,
        task="single_target_radar",
        **_,
    ):

        super().__init__()

        if task not in {"single_target_radar", "radar_benchmark"}:
            raise ValueError(
                "BiMambaTrackerModel now supports only the single-target radar task."
            )

        self.state_dim = state_dim
        self.obs_dim = obs_dim
        self.dt = dt
        if self.state_dim <= 0 or self.obs_dim <= 0:
            raise ValueError("state_dim and obs_dim must be positive integers.")
        if self.dt <= 0:
            raise ValueError("dt must be positive for the single-target radar model.")

        scalar_features = 9
        self.token_dim = scalar_features + 2 * state_dim
        if self.token_dim != 21:
            raise ValueError(
                "The single-target radar token layout expects 21 features for a 6D state."
            )

        self.input_proj = nn.Sequential(
            nn.Linear(self.token_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.SiLU(),
        )
        self.encoder = BiMambaEncoder(
            dim=embed_dim,
            num_layers=num_layers,
            d_state=d_state,
            d_conv=d_conv,
            expand=expand,
            dropout=dropout,
        )
        self.output_norm = nn.LayerNorm(embed_dim * 2)
        self.state_head = PredictionHead(
            input_dim=embed_dim * 2,
            state_dim=state_dim,
            hidden_dim=hidden_dim,
        )

    def _model_device(self):

        return next(self.parameters()).device

    def _prepare_observations(self, observations):

        device = self._model_device()
        if not isinstance(observations, torch.Tensor):
            observations = torch.tensor(observations, dtype=torch.float32, device=device)
        else:
            observations = observations.to(device=device, dtype=torch.float32)

        if observations.dim() == 2:
            observations = observations.unsqueeze(0)
        if observations.dim() != 3:
            raise ValueError(
                "observations must have shape [batch, obs_dim, seq_len] or [obs_dim, seq_len]."
            )
        if observations.shape[1] != self.obs_dim:
            raise ValueError(
                f"observations expect obs_dim={self.obs_dim}, got shape {tuple(observations.shape)}."
            )
        if observations.shape[2] < 1:
            raise ValueError("observations must contain at least one time step.")
        return observations

    def _prepare_init_state(self, init_state, batch_size):

        device = self._model_device()
        if not isinstance(init_state, torch.Tensor):
            init_state = torch.tensor(init_state, dtype=torch.float32, device=device)
        else:
            init_state = init_state.to(device=device, dtype=torch.float32)

        if init_state.dim() == 1:
            init_state = init_state.unsqueeze(0).unsqueeze(-1)
        elif init_state.dim() == 2:
            if init_state.shape[1] == self.state_dim:
                init_state = init_state.unsqueeze(-1)
            elif init_state.shape[0] == self.state_dim and init_state.shape[1] == 1:
                init_state = init_state.unsqueeze(0)
            else:
                raise ValueError(
                    "init_state with 2 dims must have shape [batch, state_dim] or [state_dim, 1]."
                )
        elif init_state.dim() != 3:
            raise ValueError(
                "init_state must have shape [batch, state_dim, 1], [batch, state_dim], or [state_dim]."
            )

        if init_state.shape[1] != self.state_dim or init_state.shape[2] != 1:
            raise ValueError(
                f"init_state expects shape [batch, {self.state_dim}, 1], got {tuple(init_state.shape)}."
            )
        if init_state.shape[0] != batch_size:
            if init_state.shape[0] == 1:
                init_state = init_state.expand(batch_size, -1, -1)
            else:
                raise ValueError(
                    "init_state batch size must match observations batch size."
                )
        return init_state

    def measurement_to_cartesian(self, observations):

        ranges = observations[:, 0, :]
        bearings = observations[:, 1, :]
        x_pos = ranges * torch.cos(bearings)
        y_pos = ranges * torch.sin(bearings)
        return torch.stack([x_pos, y_pos], dim=1)

    def coarse_state_sequence(self, observations, init_state):

        positions = self.measurement_to_cartesian(observations)
        batch_size, _, seq_len = positions.shape
        coarse = torch.zeros(
            batch_size,
            self.state_dim,
            seq_len,
            device=observations.device,
            dtype=observations.dtype,
        )

        coarse[:, 0, :] = positions[:, 0, :]
        coarse[:, 3, :] = positions[:, 1, :]

        coarse[:, 1, 0] = init_state[:, 1, 0]
        coarse[:, 4, 0] = init_state[:, 4, 0]
        if seq_len > 1:
            coarse[:, 1, 1:] = (positions[:, 0, 1:] - positions[:, 0, :-1]) / self.dt
            coarse[:, 4, 1:] = (positions[:, 1, 1:] - positions[:, 1, :-1]) / self.dt

        coarse[:, 2, 0] = init_state[:, 2, 0]
        coarse[:, 5, 0] = init_state[:, 5, 0]
        if seq_len > 1:
            coarse[:, 2, 1] = coarse[:, 2, 0]
            coarse[:, 5, 1] = coarse[:, 5, 0]
        if seq_len > 2:
            coarse[:, 2, 2:] = (coarse[:, 1, 2:] - coarse[:, 1, 1:-1]) / self.dt
            coarse[:, 5, 2:] = (coarse[:, 4, 2:] - coarse[:, 4, 1:-1]) / self.dt

        return coarse

    def build_tokens(self, observations, init_state):

        cartesian = self.measurement_to_cartesian(observations)
        coarse = self.coarse_state_sequence(observations, init_state)

        ranges = observations[:, 0, :]
        bearings = observations[:, 1, :]
        sin_bearing = torch.sin(bearings)
        cos_bearing = torch.cos(bearings)

        delta_range = torch.zeros_like(ranges)
        delta_bearing = torch.zeros_like(bearings)
        delta_x = torch.zeros_like(cartesian[:, 0, :])
        delta_y = torch.zeros_like(cartesian[:, 1, :])
        delta_range[:, 1:] = ranges[:, 1:] - ranges[:, :-1]
        delta_bearing[:, 1:] = wrap_to_pi(bearings[:, 1:] - bearings[:, :-1])
        delta_x[:, 1:] = cartesian[:, 0, 1:] - cartesian[:, 0, :-1]
        delta_y[:, 1:] = cartesian[:, 1, 1:] - cartesian[:, 1, :-1]

        init_repeat = init_state.squeeze(-1).unsqueeze(-1).expand(-1, -1, observations.shape[-1])

        token_parts = [
            ranges.unsqueeze(1),
            sin_bearing.unsqueeze(1),
            cos_bearing.unsqueeze(1),
            cartesian,
            delta_range.unsqueeze(1),
            delta_bearing.unsqueeze(1),
            delta_x.unsqueeze(1),
            delta_y.unsqueeze(1),
            coarse,
            init_repeat,
        ]
        tokens = torch.cat(token_parts, dim=1)
        if tokens.shape[1] != self.token_dim:
            raise RuntimeError(
                f"Token has {tokens.shape[1]} features but expected {self.token_dim}."
            )

        return tokens, coarse

    def forward(self, observations=None, init_state=None, history=None, measurement=None, **_):

        if observations is None:
            observations = history
        if init_state is None:
            init_state = measurement
        if observations is None or init_state is None:
            raise ValueError("Single-target radar tracking requires observations and init_state.")

        observations = self._prepare_observations(observations)
        init_state = self._prepare_init_state(init_state, observations.shape[0])

        tokens, coarse = self.build_tokens(observations, init_state)
        token_sequence = tokens.transpose(1, 2)
        coarse_sequence = coarse.transpose(1, 2)

        embedded_tokens = self.input_proj(token_sequence)
        encoded_tokens = self.encoder(embedded_tokens)
        encoded_tokens = self.output_norm(encoded_tokens)

        state_estimates = self.state_head(encoded_tokens, coarse_sequence)

        return {
            "tokens": tokens,
            "coarse_state": coarse,
            "embedded_tokens": embedded_tokens,
            "encoded_tokens": encoded_tokens,
            "state_estimates": state_estimates.transpose(1, 2),
        }


### benchmark_mamba.py

**File:** `benchmark_mamba.py`


In [ ]:
import json
import os
import random
import sys
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from data_generation import generate_synthetic_data, split_data
from data_model import ConstantAcceleration2DModel
from ekf import batch_ekf, compute_lossvstime, extended_kf
from knet_nn import KalmanNetNN
from sim_params import simparams_class, validate_simparams
from train_validate_test import evaluate_loss_knet, train_validate_knet


PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
MODEL_ROOT = os.path.join(PROJECT_ROOT, "BiMambaTracker")
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from models.tracker_model import BiMambaTrackerModel


SCALE = 1000.0
DEFAULT_BENCHMARK_DIR = str(
    Path(__file__).resolve().parent / "benchmark_outputs_project"
)
REFERENCE_SEQ_LEN = 100


@dataclass
class MambaTrainConfig:
    seed: int = 42
    epochs: int = 180
    batch_size: int = 16
    eval_batch_size: int = 16
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    patience: int = 25
    min_delta: float = 1e-6
    embed_dim: int = 128
    hidden_dim: int = 256
    num_layers: int = 3
    d_state: int = 16
    d_conv: int = 4
    expand: int = 2
    dropout: float = 0.08
    checkpoint_name: str = "best_mamba_tracker.pt"


def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def env_int(name, default=None, *, minimum=None):

    raw = os.environ.get(name)
    if raw is None:
        return default
    value = int(raw)
    if minimum is not None and value < minimum:
        raise ValueError(f"{name} must be >= {minimum}, got {value}.")
    return value


def env_float(name, default=None, *, minimum=None):

    raw = os.environ.get(name)
    if raw is None:
        return default
    value = float(raw)
    if minimum is not None and value < minimum:
        raise ValueError(f"{name} must be >= {minimum}, got {value}.")
    return value


def resolve_benchmark_dir():

    benchmark_dir = os.environ.get("BENCHMARK_DIR", DEFAULT_BENCHMARK_DIR)
    benchmark_path = Path(benchmark_dir)
    if not benchmark_path.is_absolute():
        benchmark_path = Path(__file__).resolve().parent / benchmark_path
    benchmark_path.mkdir(parents=True, exist_ok=True)
    return str(benchmark_path)


def require_cuda():

    if not torch.cuda.is_available():
        raise RuntimeError(
            "The clean BiMamba pipeline is CUDA-only in this project because "
            "the Mamba implementation used here does not support CPU execution."
        )
    return torch.device("cuda")


def apply_mamba_env_overrides(config):

    config.seed = env_int("MAMBA_SEED", config.seed, minimum=0)
    config.epochs = env_int("MAMBA_EPOCHS", config.epochs, minimum=1)
    config.patience = env_int("MAMBA_PATIENCE", config.patience, minimum=1)
    config.batch_size = env_int("MAMBA_BATCH_SIZE", config.batch_size, minimum=1)
    config.eval_batch_size = env_int(
        "MAMBA_EVAL_BATCH_SIZE",
        config.eval_batch_size,
        minimum=1,
    )
    config.embed_dim = env_int("MAMBA_EMBED_DIM", config.embed_dim, minimum=1)
    config.hidden_dim = env_int("MAMBA_HIDDEN_DIM", config.hidden_dim, minimum=1)
    config.num_layers = env_int("MAMBA_NUM_LAYERS", config.num_layers, minimum=1)
    config.d_state = env_int("MAMBA_D_STATE", config.d_state, minimum=1)
    config.d_conv = env_int("MAMBA_D_CONV", config.d_conv, minimum=1)
    config.expand = env_int("MAMBA_EXPAND", config.expand, minimum=1)
    config.dropout = env_float("MAMBA_DROPOUT", config.dropout, minimum=0.0)
    config.learning_rate = env_float("MAMBA_LR", config.learning_rate, minimum=0.0)
    config.weight_decay = env_float(
        "MAMBA_WEIGHT_DECAY",
        config.weight_decay,
        minimum=0.0,
    )
    config.min_delta = env_float("MAMBA_MIN_DELTA", config.min_delta, minimum=0.0)
    checkpoint_name = os.environ.get("MAMBA_CHECKPOINT_NAME")
    if checkpoint_name is not None:
        checkpoint_name = checkpoint_name.strip()
        if not checkpoint_name:
            raise ValueError("MAMBA_CHECKPOINT_NAME cannot be empty.")
        config.checkpoint_name = checkpoint_name
    if not 0.0 <= config.dropout < 1.0:
        raise ValueError(f"MAMBA_DROPOUT must be in [0, 1), got {config.dropout}.")
    return config


def scale_dataset(true_state, observations, init_state, process_noise, measurement_noise):

    true_state = true_state.clone()
    observations = observations.clone()
    init_state = init_state.clone()
    process_noise = process_noise.clone()
    measurement_noise = measurement_noise.clone()

    true_state /= SCALE
    init_state /= SCALE
    process_noise /= (SCALE ** 2)
    observations[:, 0, :] /= SCALE
    measurement_noise[0, 0] /= (SCALE ** 2)

    return true_state, observations, init_state, process_noise, measurement_noise


def prepare_train_val_data(ca_model, simparams):

    true_state, observations, init_state, _, _ = generate_synthetic_data(
        ca_model,
        simparams,
        snr_dB=20
    )
    true_state, observations, init_state, _, _ = scale_dataset(
        true_state,
        observations,
        init_state,
        torch.eye(ca_model.m),
        torch.eye(ca_model.n)
    )
    true_state = true_state.cpu()
    observations = observations.cpu()
    init_state = init_state.cpu()
    return split_data(true_state, observations, init_state)


def prepare_test_data(ca_model, simparams):

    true_state, observations, init_state, process_noise, measurement_noise = generate_synthetic_data(
        ca_model,
        simparams,
        snr_dB=20,
        seq_mul_fact=1
    )
    true_state, observations, init_state, process_noise, measurement_noise = scale_dataset(
        true_state,
        observations,
        init_state,
        process_noise,
        measurement_noise
    )
    return (
        true_state.cpu(),
        observations.cpu(),
        init_state.cpu(),
        process_noise.cpu(),
        measurement_noise.cpu()
    )


def build_mamba_model(simparams, config, device):

    return BiMambaTrackerModel(
        task="radar_benchmark",
        state_dim=6,
        obs_dim=2,
        dt=simparams.dt,
        embed_dim=config.embed_dim,
        hidden_dim=config.hidden_dim,
        num_layers=config.num_layers,
        d_state=config.d_state,
        d_conv=config.d_conv,
        expand=config.expand,
        dropout=config.dropout
    ).to(device)


def evaluate_mamba(model, x_pool, y_pool, init_state, loss_fn=None, batch_size=None):

    if loss_fn is None:
        loss_fn = torch.nn.MSELoss(reduction="mean")
    if batch_size is None or batch_size >= x_pool.shape[0]:
        batch_size = x_pool.shape[0]

    model.eval()
    outputs_all = []
    loss_sum = 0.0
    count = 0
    with torch.no_grad():
        for start_idx in range(0, x_pool.shape[0], batch_size):
            end_idx = min(start_idx + batch_size, x_pool.shape[0])
            outputs = model(
                observations=y_pool[start_idx:end_idx],
                init_state=init_state[start_idx:end_idx]
            )
            x_est = outputs["state_estimates"]
            batch_target = x_pool[start_idx:end_idx]
            batch_loss = loss_fn(x_est, batch_target)
            loss_sum += batch_loss.item() * (end_idx - start_idx)
            count += end_idx - start_idx
            outputs_all.append(x_est)

    return loss_sum / max(count, 1), torch.cat(outputs_all, dim=0)


def train_mamba(model, train_loader, val_tensors, config, device, checkpoint_path):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )
    loss_fn = torch.nn.MSELoss(reduction="mean")

    best_val = float("inf")
    best_epoch = 0
    history = {"train_loss": [], "val_loss": []}
    patience_counter = 0

    for epoch in range(config.epochs):
        model.train()
        train_loss_sum = 0.0
        train_batches = 0

        for obs_batch, init_batch, state_batch in train_loader:
            obs_batch = obs_batch.to(device)
            init_batch = init_batch.to(device)
            state_batch = state_batch.to(device)

            optimizer.zero_grad()
            outputs = model(
                observations=obs_batch,
                init_state=init_batch
            )
            loss = loss_fn(outputs["state_estimates"], state_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss_sum += loss.item()
            train_batches += 1

        train_loss = train_loss_sum / max(1, train_batches)
        val_loss, _ = evaluate_mamba(
            model,
            x_pool=val_tensors["state"].to(device),
            y_pool=val_tensors["obs"].to(device),
            init_state=val_tensors["init"].to(device),
            loss_fn=loss_fn,
            batch_size=config.eval_batch_size
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        print(
            f"BiMamba Epoch {epoch + 1:03d} "
            f"| train={10 * np.log10(train_loss):.4f} dB "
            f"| val={10 * np.log10(val_loss):.4f} dB"
        )

        if val_loss < best_val - config.min_delta:
            best_val = val_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config.patience:
            print(f"Early stopping BiMamba at epoch {epoch + 1}")
            break

    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    return model, history, best_epoch, best_val


def plot_loss_curves(history, output_path):

    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, 10 * np.log10(np.asarray(history["train_loss"])), label="Train")
    plt.plot(epochs, 10 * np.log10(np.asarray(history["val_loss"])), label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("MSE (dB)")
    plt.title("BiMamba Tracker Training")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close()


def plot_mse_time(time_idx, series_dict, output_path):

    plt.figure(figsize=(10, 6))
    for label, values in series_dict.items():
        plt.plot(time_idx, values, marker="o", linewidth=2, label=label)
    plt.xlabel("Time Step")
    plt.ylabel("MSE (dB)")
    plt.title("Long-Sequence Radar Benchmark")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close()


def plot_trajectory(obs_test, ts_test, estimates, output_path):

    idx = 0
    ix = 0
    iy = 3

    gt_x = ts_test[idx, ix, :].cpu().numpy()
    gt_y = ts_test[idx, iy, :].cpu().numpy()

    r_obs = obs_test[idx, 0, :].cpu().numpy()
    theta_obs = obs_test[idx, 1, :].cpu().numpy()
    obs_x = r_obs * np.cos(theta_obs)
    obs_y = r_obs * np.sin(theta_obs)

    plt.figure(figsize=(10, 8))
    plt.scatter(obs_x, obs_y, label="Observations", alpha=0.6)
    plt.plot(gt_x, gt_y, "k-", linewidth=2, label="Ground Truth")

    styles = {
        "EKF": ("--", "green"),
        "KalmanNet": ("-.", "blue"),
        "BiMamba": (":", "red")
    }
    for label, state_est in estimates.items():
        line_style, color = styles[label]
        est_x = state_est[idx, ix, :].cpu().numpy()
        est_y = state_est[idx, iy, :].cpu().numpy()
        plt.plot(est_x, est_y, line_style, color=color, linewidth=2, label=label)

    plt.scatter(gt_x[0], gt_y[0], c="green", marker="^", s=60, label="Start")
    plt.scatter(gt_x[-1], gt_y[-1], c="red", marker="X", s=60, label="End")
    plt.xlabel("Position X (km)")
    plt.ylabel("Position Y (km)")
    plt.title("Trajectory Comparison")
    plt.grid(True)
    plt.axis("equal")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close()


def save_report(metrics, output_path, output_dir_label):

    knet_retrained = metrics["knet"]["retrained_for_current_context"]
    knet_fairness = (
        "same-context comparison"
        if knet_retrained or metrics["sequence_length"] == REFERENCE_SEQ_LEN
        else "cross-length generalization only"
    )

    lines = [
        "# Synthetic Radar Benchmark",
        "",
        "## Compared Methods",
        "- EKF baseline from the provided project",
        "- KalmanNet baseline evaluated from either the provided checkpoint or a same-context retraining run",
        "- BiMambaTrackerModel in radar-benchmark mode",
        "",
        "## Aggregate Test MSE",
        f"- EKF: {metrics['ekf']['total_mse_db']:.3f} dB",
        f"- KalmanNet: {metrics['knet']['total_mse_db']:.3f} dB",
        f"- BiMamba: {metrics['mamba']['total_mse_db']:.3f} dB",
        "",
        "## Notes",
        f"- Dataset source: {metrics['data_source']}",
        f"- Sequence length: {metrics['sequence_length']}",
        f"- Dataset size: {metrics['dataset_size']}",
        f"- Measurement type: {metrics['measurement_type']}",
        f"- Device: {metrics['device']}",
        f"- KalmanNet parameters: {metrics['knet']['param_count']}",
        f"- KalmanNet checkpoint source: {metrics['knet']['checkpoint_source']}",
        f"- KalmanNet fairness status: {knet_fairness}",
        f"- BiMamba parameters: {metrics['mamba']['param_count']}",
        "",
        "## Output Files",
        f"- {output_dir_label}/benchmark_metrics.json",
        f"- {output_dir_label}/mse_time_comparison.png",
        f"- {output_dir_label}/trajectory_comparison.png",
        f"- {output_dir_label}/mamba_training_curve.png",
    ]
    with open(output_path, "w", encoding="utf-8") as file:
        file.write("\n".join(lines) + "\n")


def count_parameters(model):

    return sum(param.numel() for param in model.parameters())


def knet_collapse_note(knet_total_db: float, reference_seq_len: int, active_seq_len: int) -> str:

    """
    Explain why a positive KalmanNet MSE can appear in the long-context setup.

    Positive dB here means the average squared estimation error is above the
    normalized 0 dB reference level. In this project, that usually indicates
    that the step-wise recurrent KalmanNet estimator did not remain stable when
    moved to a much longer context length.
    """
    if knet_total_db <= 0.0:
        return ""

    return (
        f"\n[NOTE] KalmanNet MSE = +{knet_total_db:.3f} dB.\n"
        f"  In this benchmark, a positive value means KalmanNet is no longer\n"
        f"  tracking well at seq_len={active_seq_len}.\n"
        f"  KalmanNet updates one step at a time through a recurrent hidden state,\n"
        f"  while this long-context setting is {active_seq_len/reference_seq_len:.1f}x\n"
        f"  longer than the original reference length ({reference_seq_len}).\n"
        f"  When the hidden-state dynamics are not stable enough, the learned gain\n"
        f"  estimate can drift and the filter can diverge.\n"
        f"  This is a key motivation for using a full-sequence model such as Mamba.\n"
    )


def configure_knet_training(simparams):

    long_context = int(simparams.seq_len) > 100

    default_batch = min(simparams.n_batch, 8) if long_context else simparams.n_batch
    default_lr = min(simparams.learning_rate, 5e-4) if long_context else simparams.learning_rate
    default_weight_decay = max(simparams.weight_decay, 1e-4)
    default_patience = min(simparams.stopping_criteria, 12) if long_context else simparams.stopping_criteria

    simparams.n_batch = env_int("KNET_BATCH_SIZE", default_batch, minimum=1)
    simparams.epochs = env_int("KNET_EPOCHS", simparams.epochs, minimum=1)
    simparams.learning_rate = env_float("KNET_LR", default_lr, minimum=0.0)
    simparams.weight_decay = env_float(
        "KNET_WEIGHT_DECAY",
        default_weight_decay,
        minimum=0.0,
    )
    simparams.stopping_criteria = env_int("KNET_PATIENCE", default_patience, minimum=1)


def should_train_knet(simparams):

    env_mode = os.environ.get("KNET_MODE")
    if env_mode is not None:
        mode = env_mode.strip().lower()
        if mode not in {"auto", "train", "load"}:
            raise ValueError("KNET_MODE must be one of: auto, train, load")
        if mode == "train":
            return True
        if mode == "load":
            return False

    env_train = os.environ.get("KNET_TRAIN")
    if env_train is not None:
        return env_train.strip().lower() in {"1", "true", "yes", "y"}

    return False


def resolve_knet_checkpoint_candidates(benchmark_dir, seq_len):

    env_checkpoint = os.environ.get("KNET_CHECKPOINT")
    if env_checkpoint is not None:
        raw_path = env_checkpoint.strip()
        if not raw_path:
            raise ValueError("KNET_CHECKPOINT cannot be empty when provided.")
        checkpoint_path = Path(raw_path)
        if not checkpoint_path.is_absolute():
            checkpoint_path = Path(__file__).resolve().parent / checkpoint_path
        if not checkpoint_path.exists():
            raise FileNotFoundError(
                f"KNET_CHECKPOINT does not exist: {checkpoint_path}"
            )
        return [(str(checkpoint_path), f"explicit checkpoint ({checkpoint_path.name})")]

    candidate_paths = []
    project_dir = Path(__file__).resolve().parent
    benchmark_name = f"best_knet_seq{int(seq_len)}.pt"

    candidate_paths.append(Path(benchmark_dir) / benchmark_name)
    candidate_paths.append(Path(DEFAULT_BENCHMARK_DIR) / benchmark_name)
    candidate_paths.extend(sorted(project_dir.glob(f"benchmark_outputs*/{benchmark_name}")))
    candidate_paths.append(project_dir / "best_knet.pt")

    candidates = []
    seen = set()
    for candidate_path in candidate_paths:
        candidate_path = candidate_path.resolve()
        candidate_key = str(candidate_path)
        if candidate_key in seen or not candidate_path.exists():
            continue
        seen.add(candidate_key)
        if candidate_path.name == "best_knet.pt":
            source = "legacy provided best_knet.pt"
        elif candidate_path.parent.resolve() == Path(benchmark_dir).resolve():
            source = f"same-context checkpoint ({candidate_path.name})"
        else:
            source = (
                f"same-context checkpoint ({candidate_path.name}) "
                f"from {candidate_path.parent.name}"
            )
        candidates.append((candidate_key, source))

    if candidates:
        return candidates

    raise FileNotFoundError(
        "No KalmanNet checkpoint found. Retrain with KNET_TRAIN=1 or provide "
        "KNET_CHECKPOINT=/path/to/checkpoint.pt."
    )


def select_best_knet_checkpoint(
    knet,
    device,
    ts_val,
    obs_val,
    init_val,
    simparams,
    benchmark_dir,
):

    candidates = resolve_knet_checkpoint_candidates(benchmark_dir, simparams.seq_len)
    best_entry = None

    for checkpoint_path, checkpoint_source in candidates:
        checkpoint_state = torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=True,
        )
        knet.load_state_dict(checkpoint_state)
        val_loss, _, _, _ = evaluate_loss_knet(
            knet,
            ts_val.to(device),
            obs_val.to(device),
            init_val.to(device),
            simparams,
        )
        print(
            f"KalmanNet candidate: {checkpoint_path} "
            f"| val={10 * np.log10(val_loss):.4f} dB"
        )
        if best_entry is None or val_loss < best_entry["val_loss"]:
            best_entry = {
                "path": checkpoint_path,
                "source": checkpoint_source,
                "state": checkpoint_state,
                "val_loss": val_loss,
            }

    if best_entry is None:
        raise RuntimeError("KalmanNet checkpoint selection failed.")

    knet.load_state_dict(best_entry["state"])
    return best_entry["path"], best_entry["source"], best_entry["state"]


def train_or_load_knet(
    ca_model,
    simparams,
    device,
    ts_train,
    obs_train,
    ts_val,
    obs_val,
    init_train,
    init_val,
    benchmark_dir,
):

    knet = KalmanNetNN()
    knet.NNBuild(ca_model, simparams)

    train_knet = should_train_knet(simparams)
    checkpoint_path = None
    checkpoint_source = ""

    if train_knet:
        checkpoint_path = os.path.join(benchmark_dir, f"best_knet_seq{int(simparams.seq_len)}.pt")
        print(f"Training KalmanNet for seq_len={simparams.seq_len}...")
        knet, _ = train_validate_knet(
            ca_model,
            knet,
            simparams,
            ts_train.to(device),
            obs_train.to(device),
            ts_val.to(device),
            obs_val.to(device),
            init_train.to(device),
            init_val.to(device),
            checkpoint_path,
        )
        if os.path.exists(checkpoint_path):
            knet_state = torch.load(checkpoint_path, map_location=device, weights_only=True)
            knet.load_state_dict(knet_state)
            checkpoint_source = f"retrained checkpoint ({os.path.basename(checkpoint_path)})"
        else:
            knet_state = knet.state_dict()
            checkpoint_source = "training attempted, no stable checkpoint was saved"
    else:
        checkpoint_path, checkpoint_source, knet_state = select_best_knet_checkpoint(
            knet,
            device,
            ts_val,
            obs_val,
            init_val,
            simparams,
            benchmark_dir,
        )
        print(f"Loading KalmanNet checkpoint: {checkpoint_path}")
        if (
            int(simparams.seq_len) != REFERENCE_SEQ_LEN
            and f"seq{int(simparams.seq_len)}" not in os.path.basename(checkpoint_path)
        ):
            print(
                f"\n[KalmanNet FAIRNESS WARNING]\n"
                f"  Loading a checkpoint trained at seq_len={REFERENCE_SEQ_LEN}\n"
                f"  but evaluating it at seq_len={simparams.seq_len}.\n"
                f"  This is not a fair same-context comparison.\n"
                f"  The resulting KalmanNet score reflects cross-length generalization,\n"
                f"  not best KalmanNet performance at this sequence length.\n"
                f"  Re-run with KNET_TRAIN=1 to retrain KalmanNet for the active context.\n"
            )

    return knet, knet_state, checkpoint_path, checkpoint_source, train_knet


def main():

    config = apply_mamba_env_overrides(MambaTrainConfig())
    benchmark_dir = resolve_benchmark_dir()
    set_seed(config.seed)

    simparams = simparams_class()
    validate_simparams(simparams)
    configure_knet_training(simparams)

    device = require_cuda()
    ca_model = ConstantAcceleration2DModel(simparams)

    (ts_train, obs_train, init_train), (ts_val, obs_val, init_val) = prepare_train_val_data(
        ca_model,
        simparams
    )
    ts_test, obs_test, init_test, q_test, r_test = prepare_test_data(ca_model, simparams)

    train_dataset = TensorDataset(obs_train, init_train, ts_train)
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        pin_memory=True,
    )
    val_tensors = {"obs": obs_val, "init": init_val, "state": ts_val}

    mamba_model = build_mamba_model(simparams, config, device)
    mamba_checkpoint = os.path.join(benchmark_dir, config.checkpoint_name)
    mamba_model, mamba_history, best_epoch, best_val = train_mamba(
        mamba_model,
        train_loader,
        val_tensors,
        config,
        device,
        mamba_checkpoint
    )
    plot_loss_curves(
        mamba_history,
        os.path.join(benchmark_dir, "mamba_training_curve.png")
    )

    knet, knet_state, knet_checkpoint, knet_checkpoint_source, knet_was_retrained = train_or_load_knet(
        ca_model,
        simparams,
        device,
        ts_train,
        obs_train,
        ts_val,
        obs_val,
        init_train,
        init_val,
        benchmark_dir,
    )

    ekf = extended_kf(ca_model, q_test / simparams.q2_scale, r_test / simparams.r2_scale, simparams)
    batch_ekf_eval = batch_ekf(ekf)
    ekf_estimates, _, _, _, _ = batch_ekf_eval.evaluate_batch(ts_test, obs_test, init_test)
    ekf_loss_time, ekf_total_db = compute_lossvstime(ts_test, ekf_estimates)

    _, knet_estimates, k_fro, ln_innov = evaluate_loss_knet(
        knet,
        ts_test.to(device),
        obs_test.to(device),
        init_test.to(device),
        simparams
    )
    knet_estimates = knet_estimates.detach().cpu()
    knet_loss_time, knet_total_db = compute_lossvstime(ts_test.cpu(), knet_estimates)
    knet_note = knet_collapse_note(float(knet_total_db), REFERENCE_SEQ_LEN, int(simparams.seq_len))

    _, mamba_estimates = evaluate_mamba(
        mamba_model,
        x_pool=ts_test.to(device),
        y_pool=obs_test.to(device),
        init_state=init_test.to(device),
        batch_size=config.eval_batch_size
    )
    mamba_estimates = mamba_estimates.detach().cpu()
    mamba_loss_time, mamba_total_db = compute_lossvstime(ts_test.cpu(), mamba_estimates)

    time_idx = list(range(ts_test.shape[2]))
    plot_mse_time(
        time_idx,
        {
            f"EKF ({float(ekf_total_db):.2f} dB)": ekf_loss_time.cpu().numpy(),
            f"KalmanNet ({float(knet_total_db):.2f} dB)": knet_loss_time.cpu().numpy(),
            f"BiMamba ({float(mamba_total_db):.2f} dB)": mamba_loss_time.cpu().numpy(),
        },
        os.path.join(benchmark_dir, "mse_time_comparison.png")
    )
    plot_trajectory(
        obs_test.cpu(),
        ts_test.cpu(),
        {
            "EKF": ekf_estimates.cpu(),
            "KalmanNet": knet_estimates,
            "BiMamba": mamba_estimates
        },
        os.path.join(benchmark_dir, "trajectory_comparison.png")
    )

    metrics_path = os.path.join(benchmark_dir, "benchmark_metrics.json")
    report_path = os.path.join(benchmark_dir, "benchmark_report.md")

    metrics = {
        "device": str(device),
        "data_source": "synthetic data generated by data_generation.py from the provided longseq code",
        "measurement_type": simparams.meas_type,
        "sequence_length": int(simparams.seq_len),
        "dataset_size": int(simparams.dataset_size),
        "mamba_train_config": asdict(config),
        "mamba_training": {
            "best_epoch": int(best_epoch),
            "best_val_loss": float(best_val),
            "best_val_loss_db": float(10 * np.log10(best_val))
        },
        "paths": {
            "benchmark_dir": benchmark_dir,
            "metrics_path": metrics_path,
            "report_path": report_path,
            "mamba_checkpoint": mamba_checkpoint,
        },
        "ekf": {
            "total_mse_db": float(ekf_total_db),
            "loss_vs_time_db": [float(value) for value in ekf_loss_time]
        },
        "knet": {
            "total_mse_db": float(knet_total_db),
            "loss_vs_time_db": [float(value) for value in knet_loss_time],
            "param_count": int(sum(value.numel() for value in knet_state.values())),
            "checkpoint": knet_checkpoint,
            "checkpoint_source": knet_checkpoint_source,
            "retrained_for_current_context": bool(knet_was_retrained),
            "kalman_gain_frobenius": [float(value) for value in k_fro],
            "log_innovation_norm": [float(value) for value in ln_innov]
        },
        "mamba": {
            "total_mse_db": float(mamba_total_db),
            "loss_vs_time_db": [float(value) for value in mamba_loss_time],
            "param_count": int(count_parameters(mamba_model))
        }
    }

    with open(metrics_path, "w", encoding="utf-8") as file:
        json.dump(metrics, file, indent=2)

    save_report(metrics, report_path, benchmark_dir)

    print("Benchmark complete")
    print("EKF total MSE (dB):", float(ekf_total_db))
    print("KalmanNet total MSE (dB):", float(knet_total_db))
    if knet_note:
        print(knet_note, end="")
    print("BiMamba total MSE (dB):", float(mamba_total_db))
    print("Saved outputs to", benchmark_dir)


if __name__ == "__main__":
    main()


## Utilities And Reproduction

These files support figure generation, explanation material, and the compact reproduction command for the active single-target workflow.


### make_presentation_figure.py

**File:** `tools/make_presentation_figure.py`


In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt


ROOT = Path("/home/spcom41/BiMambaTracker/radar_longseq_benchmark/Radar_Tracking_Benchmark")
BENCHMARK_DIR = Path(os.environ.get("BENCHMARK_DIR", "benchmark_outputs_project"))
if not BENCHMARK_DIR.is_absolute():
    BENCHMARK_DIR = ROOT / BENCHMARK_DIR

METRICS_PATH = BENCHMARK_DIR / "benchmark_metrics.json"
OUTPUT_PATH = BENCHMARK_DIR / "presentation_result_summary.png"


def main() -> None:
    with METRICS_PATH.open() as f:
        metrics = json.load(f)

    methods = ["EKF", "KalmanNet", "Mamba"]
    values = [
        metrics["ekf"]["total_mse_db"],
        metrics["knet"]["total_mse_db"],
        metrics["mamba"]["total_mse_db"],
    ]

    colors = ["#4f5d75", "#ef8354", "#2d6a4f"]
    best_idx = min(range(len(values)), key=values.__getitem__)
    y_min = min(values)
    y_max = max(values)
    padding = max(2.0, 0.12 * (y_max - y_min))

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, ax = plt.subplots(figsize=(10, 6.5))

    bars = ax.bar(methods, values, color=colors, width=0.62, edgecolor="black", linewidth=0.8)
    bars[best_idx].set_hatch("//")

    ax.set_title("Synthetic Radar Benchmark: Aggregate Test MSE", fontsize=16, weight="bold")
    ax.set_ylabel("MSE (dB, lower is better)", fontsize=12)
    ax.set_ylim(y_min - padding, y_max + padding)
    ax.axhline(values[1], color=colors[1], linestyle="--", linewidth=1.4, alpha=0.7)
    ax.axhline(values[2], color=colors[2], linestyle="--", linewidth=1.4, alpha=0.7)

    for i, (bar, value) in enumerate(zip(bars, values)):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.3,
            f"{value:.3f} dB",
            ha="center",
            va="bottom",
            fontsize=11,
            weight="bold",
        )
        if i == best_idx:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value - max(0.9, padding * 0.25),
                "Best",
                ha="center",
                va="top",
                fontsize=11,
                color="white",
                weight="bold",
            )

    knet_gain = metrics["knet"]["total_mse_db"] - metrics["mamba"]["total_mse_db"]
    ekf_gain = metrics["ekf"]["total_mse_db"] - metrics["mamba"]["total_mse_db"]
    note = (
        f"Mamba improves over KalmanNet by {knet_gain:.3f} dB\n"
        f"Mamba improves over EKF by {ekf_gain:.3f} dB\n"
        f"Sequence length = {metrics['sequence_length']}, "
        f"Dataset size = {metrics['dataset_size']}, "
        f"Measurement = {metrics['measurement_type']}"
    )
    ax.text(
        0.98,
        0.98,
        note,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=9.5,
        bbox={"boxstyle": "round,pad=0.4", "facecolor": "white", "edgecolor": "#333333"},
    )

    fig.tight_layout()
    fig.savefig(OUTPUT_PATH, dpi=220, bbox_inches="tight")
    print(f"Saved {OUTPUT_PATH}")


if __name__ == "__main__":
    main()


### run_full_benchmark.sh

**File:** `run_full_benchmark.sh`


In [ ]:
#!/usr/bin/env bash
set -euo pipefail

ROOT="/home/spcom41/BiMambaTracker/radar_longseq_benchmark/Radar_Tracking_Benchmark"
PY="/home/spcom41/BiMambaTracker/.venv/bin/python"

cd "$ROOT"

echo "[1/2] Running synthetic single-target benchmark..."
BENCHMARK_DIR=benchmark_outputs_project \
SIM_SEQ_LEN=100 \
KNET_TRAIN=1 \
MAMBA_SEED=42 \
MAMBA_EPOCHS=180 \
MAMBA_EMBED_DIM=160 \
MAMBA_HIDDEN_DIM=320 \
MAMBA_NUM_LAYERS=3 \
MAMBA_DROPOUT=0.08 \
MAMBA_LR=0.001 \
MAMBA_WEIGHT_DECAY=0.0001 \
MAMBA_BATCH_SIZE=16 \
MAMBA_EVAL_BATCH_SIZE=16 \
"$PY" benchmark_mamba.py

echo "[2/2] Creating presentation summary figure..."
"$PY" tools/make_presentation_figure.py

echo
echo "Synthetic single-target benchmark workflow completed."
echo "Outputs: $ROOT/benchmark_outputs_project"


### architecture_verification.md


# Architecture Verification Note

This note verifies the implemented BiMambaTracker architecture against the code in this repository.

## Verified Scope

The active project is:

- single-target radar tracking
- synthetic-data benchmarking from the provided `longseq` code
- range-bearing observation processing
- Bi-Mamba temporal sequence encoding
- residual 6D state correction

The active architecture diagram is:

- `project_architecture_diagram.svg`

## Code-to-Architecture Mapping

### 1. Input and coarse state construction

Implemented in:

- `../../BiMambaTracker/models/tracker_model.py`

Verified blocks:

- radar observations `y_1:T = [r_t, theta_t]`
- initial state `x_init`
- Cartesian conversion from range and bearing
- finite-difference coarse state sequence for position, velocity, and acceleration

### 2. Tokenization and embedding

Implemented in:

- `../../BiMambaTracker/models/tracker_model.py`

Verified token content:

- range
- sine and cosine of bearing
- Cartesian position
- delta range
- delta bearing
- delta x
- delta y
- coarse 6D state
- repeated initial 6D state

This produces the verified 21D token layout used by the model.

### 3. Bi-Mamba temporal encoder

Implemented in:

- `../../BiMambaTracker/models/bimamba_encoder.py`

Verified blocks:

- forward residual Mamba stack
- backward residual Mamba stack on reversed sequence
- concatenation of forward and backward features
- fused layer normalization

### 4. Residual prediction head

Implemented in:

- `../../BiMambaTracker/models/prediction_head.py`

Verified behavior:

- an MLP predicts a residual correction from encoded features
- the correction is added to the coarse state sequence
- final output is `state_estimates in R^(6 x T)`

## What Is Not Implemented

The following elements are not part of the active project:

- multi-target trajectory prediction
- data association
- refinement head using associated measurements
- track management
- track initiation/termination logic

## Verification Result

The repository implements a clean single-target residual Bi-Mamba tracker.

The attached multi-target MambaTrack-style figure does not match the codebase and should not be used
as the main architecture figure for this project. The updated `project_architecture_diagram.svg`
matches the implemented architecture much more closely.


## Archived Note

Files such as `multitarget_data.py`, `multitarget_trackers.py`, `benchmark_multitarget.py`, `association_model.py`, and `train_association.py` remain in the folder as archived exploration. The current implementation notebook focuses only on the active single-target benchmark path.
